# Document Restoration U-Net v5 (Huấn luyện Nâng cao với Dữ liệu Destroyer + Real Pairs)

Notebook này thực thi huấn luyện mô hình **`DocumentRestorationUNet v5`**:
- **Đầu vào Đa tỉ lệ Bền vững (Multi-Scale Robust Input - 3 kênh)**: `[x, x_norm_robust_31, x_norm_robust_101]` với bộ lọc trung vị 3x3 khử xung nhiễu muối tiêu.
- **GroupNorm (8 groups)**: Thay thế toàn bộ BatchNorm2d để hoạt động ổn định không bị lệch sáng tile 512x512.
- **Bilinear Upsampling + Conv3x3**: Thay thế ConvTranspose2d triệt tiêu hoàn toàn lỗi răng cưa bàn cờ.
- **Bottleneck Đa tầm nhìn**: 1 ResidualBlock(512) kết hợp ASPP(512) (dilations 1, 2, 4, 8 + Global Average Pooling).
- **Skip Connections Chọn lọc**: AttentionGate trên tầng e2 & e3 (lọc bỏ bóng mờ/thấu mực); Skip trực tiếp trên e1 (bảo vệ 100% dấu Tiếng Việt).
- **Logit-Space Residual Output Head**: Khởi tạo $\Delta = 0 \implies \hat{y} \approx x_{\text{norm\_robust\_31}}$, gradient mượt trên toàn miền $(0, 1)$.
- **Hàm Loss DocumentCleanLoss v5**: Weighted Charbonnier (1.0) + Multi-Scale Sobel (0.8) + SSIM (0.2) + Asymmetric Stroke Darkening (0.25) + Bleed-Through Suppression (0.15) + FFT Frequency L1 (0.05).
- **Tập dữ liệu Huấn luyện**: 600 cặp Destroyer + 201 mẫu Real Pairs (67 ảnh thực tế x3 oversample) + 4 ảnh Real trong tập Val + 4 ảnh Real trong tập Test (0% data leakage).


In [ ]:
#@title 1. Mount Google Drive & Kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, os, sys, zipfile, shutil, importlib, glob
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Hãy vào Runtime > Change runtime type > T4 GPU)')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
#@title 2. Cấu hình đường dẫn, Giải nén datasets_colab_with_real.zip & Chế độ Warm-Start 30 Epochs
PROJECT_DIR = '/content/drive/MyDrive/doc-force-ai'               #@param {type:"string"}
ZIP_PATH    = '/content/drive/MyDrive/datasets_colab_with_real.zip' #@param {type:"string"}
FORCE_UNZIP = True                                                #@param {type:"boolean"}
RUN_NAME    = 'unet_v5_destroyer_real'                          #@param {type:"string"}
IMAGE_SIZE  = 512     #@param {type:"integer"}
BATCH_SIZE  = 4       #@param {type:"integer"}
EPOCHS      = 40                                                #@param {type:"integer"}
LR          = 2e-4                                              #@param {type:"number"}
VIS_EVERY   = 5       #@param {type:"integer"}
SEED        = 42

os.makedirs(PROJECT_DIR, exist_ok=True)
DATASET_ROOT = f'{PROJECT_DIR}/datasets'
MANIFEST     = f'{DATASET_ROOT}/metadata/dataset_manifest.csv'
TRAIN_CSV    = f'{DATASET_ROOT}/metadata/train.csv'
VAL_CSV      = f'{DATASET_ROOT}/metadata/val.csv'
TEST_CSV     = f'{DATASET_ROOT}/metadata/test.csv'
RUN_DIR      = f'{PROJECT_DIR}/runs/{RUN_NAME}'
MODELS_DIR   = f'{PROJECT_DIR}/models'
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

candidate_zips = [
    ZIP_PATH,
    '/content/drive/MyDrive/datasets_colab_with_real.zip',
    f'{PROJECT_DIR}/datasets_colab_with_real.zip',
    '/content/datasets_colab_with_real.zip',
    '/content/drive/MyDrive/datasets_colab_v3.zip',
    f'{PROJECT_DIR}/datasets_colab_v3.zip',
    '/content/datasets_colab_v3.zip',
]
found_zip = next((z for z in candidate_zips if os.path.exists(z)), None)

target_check_files = [
    f'{DATASET_ROOT}/processed/test/target/Anh11.jfif',
    f'{DATASET_ROOT}/processed/real_pairs/target/real_01.png',
    f'{DATASET_ROOT}/processed/real_pairs/target/real_20.png',
    f'{DATASET_ROOT}/processed/test/input/real_11.jpg',
    f'{DATASET_ROOT}/processed/val/input/real_12.jpg',
]
need_unzip = FORCE_UNZIP or (not os.path.exists(TEST_CSV)) or any(not os.path.exists(p) for p in target_check_files)

if need_unzip and found_zip:
    print(f'Đang giải nén dữ liệu mới nhất từ {found_zip} vào {PROJECT_DIR} ...')
    with zipfile.ZipFile(found_zip, 'r') as zf:
        zf.extractall(PROJECT_DIR)
    print('Giải nén hoàn tất!')
elif os.path.exists(MANIFEST):
    print(f'Đã tìm thấy dữ liệu tại: {DATASET_ROOT}')
else:
    raise FileNotFoundError(f'Không tìm thấy {MANIFEST} hoặc file zip tại {ZIP_PATH}. Hãy upload datasets_colab_with_real.zip lên Drive.')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())




In [ ]:
#@title 3. Tự động cập nhật bộ mã nguồn mới nhất vào thư mục training/
os.makedirs(f'{PROJECT_DIR}/training', exist_ok=True)

RESTORATION_DATASET_CODE = 'import os\n\nimport csv\n\nimport logging\n\nimport random\n\nfrom typing import Tuple, List, Optional, Dict\n\nfrom PIL import Image\n\nimport numpy as np\n\nimport cv2\n\nimport torch\n\nfrom torch.utils.data import Dataset\n\nimport torchvision.transforms as T\n\n\n\nlogger = logging.getLogger("restoration_dataset")\n\n\n\n\n\ndef load_image(path: str) -> Optional[np.ndarray]:\n\n    """\n\n    Safely load an image supporting unicode paths and all common formats (.png, .jpg, .jfif, .webp).\n\n    Returns BGR numpy array or None on failure.\n\n    """\n\n    if not os.path.exists(path):\n\n        return None\n\n    try:\n\n        pil_img = Image.open(path).convert("RGB")\n\n        img_rgb = np.array(pil_img)\n\n        return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)\n\n    except Exception:\n\n        try:\n\n            return cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)\n\n        except Exception:\n\n            return None\n\n\n\n\n\ndef save_image(path: str, img: np.ndarray) -> bool:\n\n    """\n\n    Save an image safely supporting unicode paths.\n\n    """\n\n    ext = os.path.splitext(path)[1]\n\n    result, buf = cv2.imencode(ext if ext else ".png", img)\n\n    if result:\n\n        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)\n\n        with open(path, "wb") as f:\n\n            f.write(buf)\n\n        return True\n\n    return False\n\n\n\n\n\ndef unsharp_mask(img: np.ndarray, amount: float = 0.8, radius: float = 1.0) -> np.ndarray:\n\n    """\n\n    Apply Unsharp Masking filter to sharpen text edges and contrast.\n\n    """\n\n    blurred = cv2.GaussianBlur(img, (0, 0), radius)\n\n    sharpened = cv2.addWeighted(img, 1.0 + amount, blurred, -amount, 0)\n\n    return np.clip(sharpened, 0, 255).astype(np.uint8)\n\n\n\n\n\ndef resize_with_padding(\n\n    img: np.ndarray,\n\n    target_size: Tuple[int, int] = (512, 512)\n\n) -> Tuple[np.ndarray, Tuple[int, int, int, int]]:\n\n    """\n\n    Resize image keeping aspect ratio and pad with zeros to reach target_size.\n\n\n\n    Returns:\n\n        canvas: padded image of shape (target_h, target_w, 3)\n\n        pad_info: (pad_top, pad_left, new_h, new_w) for later unpadding\n\n    """\n\n    h, w = img.shape[:2]\n\n    target_h, target_w = target_size\n\n    scale = min(target_w / w, target_h / h)\n\n    new_w, new_h = int(w * scale), int(h * scale)\n\n\n\n    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)\n\n\n\n    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)\n\n    pad_top = (target_h - new_h) // 2\n\n    pad_left = (target_w - new_w) // 2\n\n    canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized\n\n\n\n    return canvas, (pad_top, pad_left, new_h, new_w)\n\n\n\n\n\ndef remove_padding(\n\n    img: np.ndarray,\n\n    pad_info: Tuple[int, int, int, int],\n\n    original_size: Tuple[int, int]\n\n) -> np.ndarray:\n\n    """\n\n    Remove padding and resize back to original dimensions.\n\n\n\n    Args:\n\n        img: padded image of shape (target_h, target_w, 3)\n\n        pad_info: (pad_top, pad_left, new_h, new_w) from resize_with_padding\n\n        original_size: (orig_h, orig_w) to resize back to\n\n    """\n\n    pad_top, pad_left, new_h, new_w = pad_info\n\n    cropped = img[pad_top:pad_top + new_h, pad_left:pad_left + new_w]\n\n    orig_h, orig_w = original_size\n\n    restored = cv2.resize(cropped, (orig_w, orig_h), interpolation=cv2.INTER_LANCZOS4)\n\n    return restored\n\n\n\n\n\ndef pad_if_needed(img: np.ndarray, patch_size: Tuple[int, int]) -> np.ndarray:\n\n    """Pad image using BORDER_REFLECT if any dimension is smaller than patch_size."""\n\n    h, w = img.shape[:2]\n\n    patch_h, patch_w = patch_size\n\n    pad_h = max(0, patch_h - h)\n\n    pad_w = max(0, patch_w - w)\n\n    if pad_h > 0 or pad_w > 0:\n\n        pad_top = pad_h // 2\n\n        pad_bottom = pad_h - pad_top\n\n        pad_left = pad_w // 2\n\n        pad_right = pad_w - pad_left\n\n        return cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REFLECT)\n\n    return img\n\n\n\n\n\ndef crop_patch(\n\n    deg_img: np.ndarray,\n\n    clean_img: np.ndarray,\n\n    patch_size: Tuple[int, int] = (512, 512),\n\n    is_train: bool = True\n\n) -> Tuple[np.ndarray, np.ndarray]:\n\n    """\n\n    Extract matching patch from degraded and clean images.\n\n    Uses RandomCrop when is_train=True, CenterCrop when is_train=False.\n\n    Pads with BORDER_REFLECT if image is smaller than patch_size in either dimension.\n\n    """\n\n    deg_pad = pad_if_needed(deg_img, patch_size)\n\n    clean_pad = pad_if_needed(clean_img, patch_size)\n\n\n\n    h, w = deg_pad.shape[:2]\n\n    patch_h, patch_w = patch_size\n\n\n\n    if is_train:\n\n        max_top = max(0, h - patch_h)\n\n        max_left = max(0, w - patch_w)\n\n        top = random.randint(0, max_top) if max_top > 0 else 0\n\n        left = random.randint(0, max_left) if max_left > 0 else 0\n\n    else:\n\n        top = max(0, (h - patch_h) // 2)\n\n        left = max(0, (w - patch_w) // 2)\n\n\n\n    deg_crop = deg_pad[top:top + patch_h, left:left + patch_w]\n\n    clean_crop = clean_pad[top:top + patch_h, left:left + patch_w]\n\n    return deg_crop, clean_crop\n\n\n\n\n\nGEOMETRIC_DISTORTION_KEYWORDS = (\n\n    "bookbinding",\n\n    "perspective",\n\n    "opticaldistortion",\n\n    "centercrop",\n\n    "bindingsandfasteners",\n\n)\n\n\n\n\n\ndef _sauvola_threshold_integral(\n\n    img_f: np.ndarray,\n\n    window_size: int = 15,\n\n    k: float = 0.15,\n\n    r: float = 128.0,\n\n) -> np.ndarray:\n\n    """\n\n    Compute local Sauvola threshold T(x,y) = mu(x,y) * [1 + k * (sigma(x,y) / R - 1)]\n\n    in O(1) time per pixel using OpenCV integral images.\n\n    """\n\n    pad = window_size // 2\n\n    padded = cv2.copyMakeBorder(img_f, pad, pad, pad, pad, cv2.BORDER_REFLECT).astype(np.float64)\n\n    s, s2 = cv2.integral2(padded)\n\n    w = window_size\n\n    area = float(w * w)\n\n\n\n    win_sum = s[w:, w:] - s[:-w, w:] - s[w:, :-w] + s[:-w, :-w]\n\n    win_sqsum = s2[w:, w:] - s2[:-w, w:] - s2[w:, :-w] + s2[:-w, :-w]\n\n\n\n    mu = win_sum / area\n\n    var = np.maximum(win_sqsum / area - (mu * mu), 0.0)\n\n    sigma = np.sqrt(var)\n\n\n\n    thresh = mu * (1.0 + k * ((sigma / r) - 1.0))\n\n    return thresh.astype(np.float32)\n\n\n\n\n\ndef clean_target_normalization(\n\n    clean_gray: np.ndarray,\n\n    use_nlm: bool = False,\n\n    gamma: float = 0.75,\n\n) -> np.ndarray:\n\n    """\n\n    Master Ground-Truth Generator: 5-step Soft Anti-Aliased Document Mastering\n\n    ("Multi-Scale Morphological + Sauvola Soft Matting").\n\n\n\n    1. Edge-Preserving Pre-Denoising (Bilateral Filter or Fast Non-Local Means).\n\n    2. Multi-Scale Morphological Background Estimation (21x21 & 45x45 elliptical closing + Gaussian sigma=15).\n\n    3. Multi-Scale Sauvola Soft Alpha Matting (O(1) integral image with w=15,k=0.15 & w=31,k=0.20 + Soft Ramp).\n\n    4. Topology-Aware Micro-Speckle Removal (removes isolated dust <= 2px with min_intensity > 140).\n\n    5. Foreground Unsharp Boost (sharpens only stroke pixels < 245).\n\n    """\n\n    if clean_gray is None or clean_gray.size == 0:\n\n        return clean_gray\n\n    if clean_gray.ndim == 3:\n\n        clean_gray = cv2.cvtColor(clean_gray, cv2.COLOR_BGR2GRAY)\n\n\n\n    # Step 1: Edge-Preserving Pre-Denoising\n\n    if use_nlm:\n\n        denoised = cv2.fastNlMeansDenoising(\n\n            clean_gray, None, h=6, templateWindowSize=7, searchWindowSize=21\n\n        )\n\n    else:\n\n        denoised = cv2.bilateralFilter(clean_gray, d=5, sigmaColor=25, sigmaSpace=25)\n\n\n\n    # Step 2: Multi-Scale Morphological Background Estimation\n\n    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))\n\n    kernel_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))\n\n    bg_small = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, kernel_small)\n\n    bg_large = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, kernel_large)\n\n    bg = np.maximum(bg_small, bg_large)\n\n    bg = cv2.GaussianBlur(bg, (0, 0), sigmaX=15, sigmaY=15)\n\n\n\n    norm = np.clip(\n\n        (denoised.astype(np.float32) / np.maximum(bg.astype(np.float32), 15.0)) * 255.0,\n\n        0.0,\n\n        255.0,\n\n    )\n\n\n\n    # Step 3: Multi-Scale Sauvola Soft Alpha Matting\n\n    t_small = _sauvola_threshold_integral(norm, window_size=15, k=0.15, r=128.0)\n\n    t_med = _sauvola_threshold_integral(norm, window_size=31, k=0.20, r=128.0)\n\n    t_sauvola = np.maximum(t_small, t_med)\n\n\n\n    diff = norm - t_sauvola\n\n    bg_mask = (diff > 6.0) | (norm > 225.0)\n\n    fg_mask = ~bg_mask\n\n\n\n    matted_f = np.full_like(norm, 255.0, dtype=np.float32)\n\n    if np.any(fg_mask):\n\n        fg_vals = norm[fg_mask]\n\n        black_floor = float(np.clip(np.percentile(fg_vals, 18), 20.0, 145.0))\n\n        upper_bound = np.minimum(t_sauvola + 6.0, 225.0)\n\n        span = np.maximum(upper_bound - black_floor, 20.0)\n\n\n\n        # Compute foreground opacity alpha in [0, 1], boost with gamma < 1.0, and map to [0, 255]\n\n        alpha = np.clip((upper_bound - norm) / span, 0.0, 1.0)\n\n        alpha_boosted = np.power(alpha, gamma)\n\n        matted_f = np.where(bg_mask, 255.0, 255.0 * (1.0 - alpha_boosted))\n\n\n\n    matted = np.clip(matted_f, 0.0, 255.0).astype(np.uint8)\n\n\n\n    # Step 4: Topology-Aware Micro-Speckle Removal\n\n    dark_mask = (matted < 200).astype(np.uint8)\n\n    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dark_mask, connectivity=8)\n\n    for lbl in range(1, num_labels):\n\n        area = stats[lbl, cv2.CC_STAT_AREA]\n\n        if area <= 2:\n\n            x0 = stats[lbl, cv2.CC_STAT_LEFT]\n\n            y0 = stats[lbl, cv2.CC_STAT_TOP]\n\n            w0 = stats[lbl, cv2.CC_STAT_WIDTH]\n\n            h0 = stats[lbl, cv2.CC_STAT_HEIGHT]\n\n            roi_labels = labels[y0:y0 + h0, x0:x0 + w0]\n\n            roi_img = matted[y0:y0 + h0, x0:x0 + w0]\n\n            comp_mask = roi_labels == lbl\n\n            if int(np.min(roi_img[comp_mask])) > 140:\n\n                roi_img[comp_mask] = 255\n\n\n\n    # Step 5: Foreground Unsharp Boost (only on stroke pixels < 245)\n\n    sharpened = unsharp_mask(matted, amount=0.65, radius=1.0)\n\n    out = np.where(matted < 245, sharpened, 255).astype(np.uint8)\n\n    return out\n\n\n\n\n\n# Alias for backward compatibility across training and evaluation\n\nnormalize_clean_target = clean_target_normalization\n\n\n\n\n\ndef apply_hard_augmentation(\n\n    deg_patch: np.ndarray,\n\n    clean_patch: np.ndarray,\n\n    is_real_pair: bool = False,\n\n) -> Tuple[np.ndarray, np.ndarray]:\n\n    """\n\n    On-the-fly hard augmentation for training patches:\n\n    - 50% probability: synchronous HorizontalFlip on both degraded and clean patches.\n\n    - For synthetic/standard samples (`is_real_pair=False`), 30% probability: synthetic local shadow\n\n      patch or mild Gaussian blur on degraded patch.\n\n    - For `real_pairs` (`is_real_pair=True`), preserves authentic camera degradation (only horizontal flip).\n\n    """\n\n    # 1. 50% chance: synchronous HorizontalFlip\n\n    if random.random() < 0.5:\n\n        deg_patch = cv2.flip(deg_patch, 1)\n\n        clean_patch = cv2.flip(clean_patch, 1)\n\n\n\n    if is_real_pair:\n\n        return deg_patch, clean_patch\n\n\n\n    # 2. 30% chance: mild Gaussian Blur and/or Synthetic Local Shadow on deg_patch\n\n    if random.random() < 0.3:\n\n        aug_mode = random.choice(["blur", "shadow", "both"])\n\n        deg_f = deg_patch.astype(np.float32)\n\n\n\n        if aug_mode in ("blur", "both"):\n\n            ksize = random.choice([3, 5])\n\n            sigma = random.uniform(0.6, 1.3)\n\n            deg_f = cv2.GaussianBlur(deg_f, (ksize, ksize), sigmaX=sigma, sigmaY=sigma)\n\n\n\n        if aug_mode in ("shadow", "both"):\n\n            h, w = deg_f.shape[:2]\n\n            # Create a smooth local shadow mask darkening a random region by factor [0.45, 0.75]\n\n            shadow_mask = np.ones((h, w), dtype=np.float32)\n\n            cx = random.randint(int(w * 0.2), int(w * 0.8))\n\n            cy = random.randint(int(h * 0.2), int(h * 0.8))\n\n            rx = random.randint(int(w * 0.25), int(w * 0.6))\n\n            ry = random.randint(int(h * 0.25), int(h * 0.6))\n\n            angle = random.uniform(0, 180)\n\n            darkness = random.uniform(0.45, 0.75)\n\n            cv2.ellipse(shadow_mask, (cx, cy), (rx, ry), angle, 0, 360, darkness, -1)\n\n            shadow_mask = cv2.GaussianBlur(shadow_mask, (0, 0), sigmaX=25, sigmaY=25)\n\n            deg_f = deg_f * shadow_mask\n\n\n\n        deg_patch = np.clip(deg_f, 0, 255).astype(np.uint8)\n\n\n\n    return deg_patch, clean_patch\n\n\n\n\n\ndef infer_single_image_tiled(\n\n    model: torch.nn.Module,\n\n    img_gray: np.ndarray,\n\n    device: torch.device,\n\n    tile_size: int = 512,\n\n    overlap: int = 128,\n\n    refine_glyphs: bool = False,\n\n) -> np.ndarray:\n\n    """\n\n    Sliding Window Tiling Inference with 2D Gaussian feather blending.\n\n    Runs 100% pure neural network forward passes through DocumentRestorationUNet.\n\n    refine_glyphs: parameter kept for compatibility; no external morphological/heuristic\n\n    filter or background clipping is performed (100% pure neural network output).\n\n    """\n\n    h_orig, w_orig = img_gray.shape[:2]\n\n\n\n    # On CPU, if image is ultra-high resolution (>1800px, e.g. 9MP-12MP camera raw),\n\n    # run tiled U-Net at 1600px working scale and upsample back\n\n    if device.type == "cpu" and max(h_orig, w_orig) > 1800 and tile_size <= 512:\n\n        scale = 1600.0 / float(max(h_orig, w_orig))\n\n        w_work = max(64, int(round(w_orig * scale)))\n\n        h_work = max(64, int(round(h_orig * scale)))\n\n        img_work = cv2.resize(img_gray, (w_work, h_work), interpolation=cv2.INTER_AREA)\n\n        pred_work = infer_single_image_tiled(\n\n            model, img_work, device, tile_size=tile_size, overlap=min(overlap, 64), refine_glyphs=refine_glyphs\n\n        )\n\n        return cv2.resize(pred_work, (w_orig, h_orig), interpolation=cv2.INTER_CUBIC)\n\n\n\n    # Small image fast path: pad to multiple of 8 and run single forward pass\n\n    if h_orig <= tile_size and w_orig <= tile_size:\n\n        pad_h = (8 - (h_orig % 8)) % 8\n\n        pad_w = (8 - (w_orig % 8)) % 8\n\n        if pad_h > 0 or pad_w > 0:\n\n            img_pad = cv2.copyMakeBorder(img_gray, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)\n\n        else:\n\n            img_pad = img_gray\n\n        tensor_in = torch.from_numpy(img_pad).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0\n\n        with torch.no_grad():\n\n            pred = model(tensor_in)\n\n        out = (torch.clamp(pred, 0.0, 1.0).squeeze().cpu().numpy() * 255.0).astype(np.uint8)\n\n        return out[:h_orig, :w_orig]\n\n\n\n    # Pad any dimension smaller than tile_size\n\n    pad_h = max(0, tile_size - h_orig)\n\n    pad_w = max(0, tile_size - w_orig)\n\n    if pad_h > 0 or pad_w > 0:\n\n        img_pad = cv2.copyMakeBorder(img_gray, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)\n\n    else:\n\n        img_pad = img_gray\n\n    h_pad, w_pad = img_pad.shape[:2]\n\n\n\n    # 2D Gaussian feather blending window (size: tile_size x tile_size)\n\n    coords = np.linspace(-1.0, 1.0, tile_size, dtype=np.float32)\n\n    g1d = np.exp(-2.0 * (coords ** 2))\n\n    weight_2d = np.outer(g1d, g1d).astype(np.float32)\n\n\n\n    stride = max(1, tile_size - overlap)\n\n    y_starts = list(range(0, h_pad - tile_size + 1, stride))\n\n    if not y_starts or y_starts[-1] != h_pad - tile_size:\n\n        y_starts.append(h_pad - tile_size)\n\n\n\n    x_starts = list(range(0, w_pad - tile_size + 1, stride))\n\n    if not x_starts or x_starts[-1] != w_pad - tile_size:\n\n        x_starts.append(w_pad - tile_size)\n\n\n\n    acc_pred = np.zeros((h_pad, w_pad), dtype=np.float32)\n\n    acc_weight = np.zeros((h_pad, w_pad), dtype=np.float32)\n\n\n\n    with torch.no_grad():\n\n        for y in y_starts:\n\n            for x in x_starts:\n\n                tile = img_pad[y:y + tile_size, x:x + tile_size]\n\n                tile_t = torch.from_numpy(tile).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0\n\n                pred_t = model(tile_t)\n\n                pred_np = torch.clamp(pred_t, 0.0, 1.0).squeeze().cpu().numpy()\n\n                acc_pred[y:y + tile_size, x:x + tile_size] += pred_np * weight_2d\n\n                acc_weight[y:y + tile_size, x:x + tile_size] += weight_2d\n\n\n\n    merged = acc_pred / np.maximum(acc_weight, 1e-6)\n\n    return (np.clip(merged[:h_orig, :w_orig], 0.0, 1.0) * 255.0).astype(np.uint8)\n\n\n\n\n\n\n\ndef _resolve_csv_path(raw_p: str, metadata_csv: str, root_dir: Optional[str] = None) -> str:\n\n    if not raw_p:\n\n        return ""\n\n    if os.path.isabs(raw_p) and os.path.exists(raw_p):\n\n        return os.path.normpath(raw_p)\n\n    if os.path.exists(raw_p):\n\n        return os.path.normpath(os.path.abspath(raw_p))\n\n    if root_dir:\n\n        cand = os.path.normpath(os.path.join(root_dir, raw_p))\n\n        if os.path.exists(cand):\n\n            return cand\n\n    if metadata_csv:\n\n        csv_dir = os.path.dirname(os.path.abspath(metadata_csv))\n\n        for p in (csv_dir, os.path.dirname(csv_dir), os.path.dirname(os.path.dirname(csv_dir))):\n\n            cand = os.path.normpath(os.path.join(p, raw_p))\n\n            if os.path.exists(cand):\n\n                return cand\n\n            if raw_p.startswith(("datasets/", "datasets\\\\")):\n\n                cand2 = os.path.normpath(os.path.join(p, raw_p[9:]))\n\n                if os.path.exists(cand2):\n\n                    return cand2\n\n    return os.path.normpath(raw_p)\n\n\n\n\n\nclass DocumentRestorationDataset(Dataset):\n\n    """\n\n    Dataset loader for Document Image Restoration model.\n\n    Reads paired degraded/clean images from a CSV metadata file.\n\n    Supports Grayscale 1-channel patch-based training (RandomCrop) and validation (CenterCrop).\n\n\n\n    Two CSV formats are supported (auto-detected from the header):\n\n\n\n    1. Manifest (``datasets/metadata/dataset_manifest.csv``) - many inputs -> one target.\n\n       Columns: split, original_id, degradation_type, variant_id, input_path, target_path.\n\n       Paths are resolved relative to ``root_dir`` (default: parent of the CSV\'s folder, i.e.\n\n       the ``datasets`` directory). Input and target names are NOT assumed to match.\n\n       Use ``split`` (\'train\'/\'val\'/\'test\') and ``degradation_types`` to filter rows.\n\n\n\n    2. Legacy: columns input, target (+ optional source_id, degradation, blur_*...).\n\n    """\n\n\n\n    def __init__(\n\n        self,\n\n        metadata_csv: str,\n\n        image_size: Tuple[int, int] = (512, 512),\n\n        patch_size: Optional[Tuple[int, int]] = None,\n\n        transform: Optional[T.Compose] = None,\n\n        return_metadata: bool = False,\n\n        split: Optional[str] = None,\n\n        root_dir: Optional[str] = None,\n\n        degradation_types: Optional[List[str]] = None,\n\n        target_cache_size: int = 64,\n\n        is_train: Optional[bool] = None,\n\n        filter_geometric: Optional[bool] = None,\n\n        include_unpaired: bool = False,\n\n    ):\n\n        if patch_size is not None:\n\n            self.patch_size = patch_size\n\n        else:\n\n            self.patch_size = image_size if isinstance(image_size, tuple) else (image_size, image_size)\n\n        self.image_size = self.patch_size\n\n        self.include_unpaired = include_unpaired\n\n\n\n        # Infer effective split (\'train\', \'val\', \'test\') when split is not explicitly passed\n\n        csv_basename = os.path.basename(metadata_csv).lower() if metadata_csv else ""\n\n        if split is not None:\n\n            effective_split = split.strip().lower()\n\n        elif "test" in csv_basename:\n\n            effective_split = "test"\n\n        elif "val" in csv_basename:\n\n            effective_split = "val"\n\n        elif "train" in csv_basename:\n\n            effective_split = "train"\n\n        else:\n\n            effective_split = "train" if (is_train is None or is_train) else "val"\n\n        self.effective_split = effective_split\n\n\n\n        if is_train is not None:\n\n            self.is_train = is_train\n\n        else:\n\n            self.is_train = (effective_split == "train")\n\n\n\n        if filter_geometric is not None:\n\n            self.filter_geometric = filter_geometric\n\n        else:\n\n            self.filter_geometric = effective_split in ("train", "val")\n\n\n\n        self.return_metadata = return_metadata\n\n        self.target_cache_size = target_cache_size\n\n        self._target_cache: Dict[str, np.ndarray] = {}\n\n\n\n        # Read CSV metadata\n\n        self.pairs: List[Dict[str, str]] = []\n\n        self.unpaired_samples: List[Dict[str, str]] = []\n\n        self.total_rows = 0\n\n        self.missing_count = 0\n\n        self.filtered_count = 0\n\n        self.unpaired_count = 0\n\n\n\n        if transform is not None:\n\n            self.transform = transform\n\n        else:\n\n            self.transform = T.Compose([\n\n                T.ToTensor(),  # converts [H,W] uint8 to [1,H,W] float [0,1]\n\n            ])\n\n\n\n        if not os.path.exists(metadata_csv):\n\n            logger.error(f"Metadata CSV file not found: \'{metadata_csv}\'")\n\n            return\n\n\n\n        if root_dir is None:\n\n            root_dir = os.path.dirname(os.path.dirname(os.path.abspath(metadata_csv)))\n\n        wanted_types = set(degradation_types) if degradation_types else None\n\n\n\n        with open(metadata_csv, "r", encoding="utf-8-sig", newline="") as f:\n\n            reader = csv.DictReader(f)\n\n            is_manifest = "input_path" in (reader.fieldnames or [])\n\n            for row in reader:\n\n                if is_manifest:\n\n                    row_split = row.get("split", "").strip()\n\n                    if split is not None and row_split != split:\n\n                        continue\n\n                    deg_str = row.get("degradation_type", "").strip()\n\n                    if wanted_types is not None and deg_str not in wanted_types:\n\n                        continue\n\n                    self.total_rows += 1\n\n                    raw_in = (row.get("input_path") or "").strip()\n\n                    raw_tgt = (row.get("target_path") or "").strip()\n\n                    input_path = os.path.normpath(os.path.join(root_dir, raw_in)) if raw_in else ""\n\n                    target_path = os.path.normpath(os.path.join(root_dir, raw_tgt)) if raw_tgt else ""\n\n                    src_id = row.get("original_id", "").strip()\n\n                    active_split = (row_split or effective_split).lower()\n\n                else:\n\n                    self.total_rows += 1\n\n                    raw_in = (row.get("input") or "").strip()\n\n                    raw_tgt = (row.get("target") or "").strip()\n\n                    input_path = _resolve_csv_path(raw_in, metadata_csv, root_dir)\n\n                    target_path = _resolve_csv_path(raw_tgt, metadata_csv, root_dir)\n\n                    deg_str = (row.get("degradation") or row.get("blur_level") or "").strip()\n\n                    src_id = (row.get("source_id") or row.get("original_id") or "").strip()\n\n                    active_split = (row.get("split") or effective_split).strip().lower()\n\n\n\n                in_stem = os.path.splitext(os.path.basename(input_path))[0].lower() if input_path else ""\n\n                is_blind_real = (\n\n                    in_stem in ("real_11", "real_12")\n\n                    or deg_str.lower() in ("unpaired_real", "blind_real")\n\n                    or (raw_tgt == "" and bool(raw_in))\n\n                )\n\n\n\n                # Never allow blind evaluation images (real_11, real_12) into the train split\n\n                if active_split == "train" and (in_stem in ("real_11", "real_12") or is_blind_real):\n\n                    self.filtered_count += 1\n\n                    continue\n\n\n\n                if is_blind_real:\n\n                    if input_path and os.path.exists(input_path):\n\n                        self.unpaired_count += 1\n\n                        unpaired_rec = {\n\n                            "input": input_path,\n\n                            "target": "",\n\n                            "source_id": src_id or in_stem,\n\n                            "degradation": deg_str or "unpaired_real",\n\n                            "split": active_split,\n\n                            "is_real_pair": False,\n\n                            "is_unpaired": True,\n\n                        }\n\n                        self.unpaired_samples.append(unpaired_rec)\n\n                        if self.include_unpaired and not self.is_train:\n\n                            self.pairs.append(unpaired_rec)\n\n                    else:\n\n                        self.missing_count += 1\n\n                    continue\n\n\n\n                if not (input_path and target_path and os.path.exists(input_path) and os.path.exists(target_path)):\n\n                    self.missing_count += 1\n\n                    continue\n\n\n\n                # Filter geometric distortions (BookBinding, Perspective, OpticalDistortion) in train & val\n\n                # and filter Anh06 in train (open book on wooden table)\n\n                if self.filter_geometric and active_split in ("train", "val"):\n\n                    check_text = f"{os.path.basename(input_path)} {deg_str}".lower()\n\n                    if any(kw in check_text for kw in GEOMETRIC_DISTORTION_KEYWORDS):\n\n                        self.filtered_count += 1\n\n                        continue\n\n                    if active_split == "train":\n\n                        id_text = f"{src_id} {os.path.basename(input_path)} {os.path.basename(target_path)}".lower()\n\n                        if "anh06" in id_text:\n\n                            self.filtered_count += 1\n\n                            continue\n\n\n\n                is_real_pair = (\n\n                    "real_pairs" in input_path.replace("\\\\", "/").lower()\n\n                    or "real_pairs" in target_path.replace("\\\\", "/").lower()\n\n                    or deg_str.lower() in ("real_pairs", "realcamera")\n\n                )\n\n\n\n                is_clean_gt = (\n\n                    str(row.get("is_clean_gt", "")).lower() in ("true", "1")\n\n                    or "clean_originals" in target_path.lower()\n\n                    or "destroyer_pairs" in target_path.lower()\n\n                )\n\n\n\n                if is_manifest:\n\n                    self.pairs.append({\n\n                        "input": input_path,\n\n                        "target": target_path,\n\n                        "source_id": src_id,\n\n                        "original_id": src_id,\n\n                        "degradation": deg_str,\n\n                        "variant_id": row.get("variant_id", ""),\n\n                        "split": row.get("split", ""),\n\n                        "is_real_pair": is_real_pair,\n\n                        "is_clean_gt": is_clean_gt,\n\n                        "is_unpaired": False,\n\n                    })\n\n                else:\n\n                    self.pairs.append({\n\n                        "input": input_path,\n\n                        "target": target_path,\n\n                        "source_id": src_id,\n\n                        "degradation": deg_str,\n\n                        "is_real_pair": is_real_pair,\n\n                        "is_clean_gt": is_clean_gt,\n\n                        "is_unpaired": False,\n\n                    })\n\n\n\n        if self.missing_count > 0:\n\n            logger.warning(\n\n                f"Missing {self.missing_count}/{self.total_rows} image pair(s) in \'{metadata_csv}\'. "\n\n                f"Loaded {len(self.pairs)} valid pair(s)."\n\n            )\n\n        elif self.total_rows == 0:\n\n            logger.warning(f"Metadata CSV \'{metadata_csv}\' is empty (split={split}).")\n\n        elif self.filtered_count > 0 or self.unpaired_count > 0:\n\n            logger.info(\n\n                f"Split=\'{effective_split}\': Loaded {len(self.pairs)} paired sample(s), "\n\n                f"{self.unpaired_count} unpaired blind-eval sample(s), "\n\n                f"filtered {self.filtered_count} geometric/non-flat pair(s)."\n\n            )\n\n\n\n    def __len__(self) -> int:\n\n        return len(self.pairs)\n\n\n\n    def _load_target(self, path: str, skip_normalization: bool = False) -> np.ndarray:\n\n        """\n\n        Load full-resolution clean target image as Grayscale (cached).\n\n        For `real_pairs` targets (already carefully refined with bleed-through & watermark removal),\n\n        skips re-running simple `clean_target_normalization`.\n\n        """\n\n        cached = self._target_cache.get(path)\n\n        if cached is not None:\n\n            return cached\n\n        clean_bgr = load_image(path)\n\n        if clean_bgr is None:\n\n            raise FileNotFoundError(f"Failed to load clean target image: {path}")\n\n        clean_gray = cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2GRAY)\n\n        is_clean_gt_path = (\n\n            "clean_originals" in path.replace("\\\\", "/").lower()\n\n            or "destroyer_pairs" in path.replace("\\\\", "/").lower()\n\n            or "real_pairs" in path.replace("\\\\", "/").lower()\n\n        )\n\n        is_real_target = skip_normalization or is_clean_gt_path\n\n        if not is_real_target:\n\n            clean_gray = clean_target_normalization(clean_gray)\n\n        if len(self._target_cache) < self.target_cache_size:\n\n            self._target_cache[path] = clean_gray\n\n        return clean_gray\n\n\n\n    def __getitem__(self, idx: int):\n\n        pair = self.pairs[idx]\n\n        is_real_pair = bool(pair.get("is_real_pair", False))\n\n        is_unpaired = bool(pair.get("is_unpaired", False)) or not pair.get("target")\n\n\n\n        # Load degraded image as numpy BGR -> Grayscale\n\n        deg_bgr = load_image(pair["input"])\n\n        if deg_bgr is None:\n\n            raise FileNotFoundError(f"Failed to load degraded input image: {pair[\'input\']}")\n\n\n\n        deg_gray = cv2.cvtColor(deg_bgr, cv2.COLOR_BGR2GRAY)\n\n\n\n        if is_unpaired:\n\n            deg_patch, _ = crop_patch(deg_gray, deg_gray, patch_size=self.patch_size, is_train=False)\n\n            deg_tensor = self.transform(deg_patch)\n\n            if deg_tensor.ndim == 2:\n\n                deg_tensor = deg_tensor.unsqueeze(0)\n\n            if self.return_metadata:\n\n                return deg_tensor, None, pair\n\n            return deg_tensor, deg_tensor\n\n\n\n        is_clean_gt = bool(pair.get("is_clean_gt", False))\n\n        clean_gray = self._load_target(pair["target"], skip_normalization=is_real_pair or is_clean_gt)\n\n\n\n        # Align dimensions if input and clean target differ in resolution (e.g. LongestMaxSize)\n\n        # Always resize degraded input to match clean target resolution\n\n        if deg_gray.shape[:2] != clean_gray.shape[:2]:\n\n            deg_gray = cv2.resize(deg_gray, (clean_gray.shape[1], clean_gray.shape[0]), interpolation=cv2.INTER_CUBIC)\n\n\n\n        # Patch-based extraction at full resolution (RandomCrop for train, CenterCrop for eval)\n\n        deg_patch, clean_patch = crop_patch(deg_gray, clean_gray, patch_size=self.patch_size, is_train=self.is_train)\n\n\n\n        # On-the-fly Hard Augmentation (HorizontalFlip 50%; skip synthetic blur/shadow for real_pairs)\n\n        if self.is_train:\n\n            deg_patch, clean_patch = apply_hard_augmentation(\n\n                deg_patch, clean_patch, is_real_pair=is_real_pair\n\n            )\n\n\n\n        # Apply transforms: converts uint8 [0,255] [H,W] to float [0,1] [1,H,W]\n\n        deg_tensor = self.transform(deg_patch)\n\n        clean_tensor = self.transform(clean_patch)\n\n\n\n        # Ensure tensor is shape [1, H, W]\n\n        if deg_tensor.ndim == 2:\n\n            deg_tensor = deg_tensor.unsqueeze(0)\n\n        if clean_tensor.ndim == 2:\n\n            clean_tensor = clean_tensor.unsqueeze(0)\n\n\n\n        if self.return_metadata:\n\n            return deg_tensor, clean_tensor, pair\n\n\n\n        return deg_tensor, clean_tensor\n\n\n\n\n\n'

TRAIN_RESTORATION_CODE = '"""\nDocument Image Restoration Model Training Script (U-Net v5).\n\nArchitecture (DocumentRestorationUNet v5):\n- Multi-Scale Robust Input (3 channels): [x, x_norm_robust_31, x_norm_robust_101] with 3x3 median pre-filtering\n- Internal layers: GroupNorm (8 groups) throughout, replacing BatchNorm to eliminate tile brightness shifts\n- Bilinear Upsampling + Conv3x3 (up3, up2, up1) replacing ConvTranspose2d to eliminate checkerboard artifacts\n- Bottleneck: 1 ResidualBlock(512) followed by ASPP(512) with dilations 1, 2, 4, 8 + Global Average Pooling\n- Selective Skip Connections: AttentionGate on levels 2 and 3; direct skip connection on level 1 to preserve 100% fine accents\n- Logit-Space Residual Output Head: delta = out_conv(d1), output = sigmoid(logit(clamp(x_norm_robust_31)) + delta)\n  with zero weight/bias initialization so training starts cleanly at the normalized state.\n\nLoss (DocumentCleanLoss):\n- Weighted Charbonnier Loss (1.0)\n- Multi-Scale Sobel Edge Gradient Loss (0.8)\n- SSIM Loss (0.2)\n- Asymmetric Stroke Darkening Loss (0.25)\n- Bleed-Through Suppression Loss (0.15)\n- FFT Frequency L1 Loss (0.05)\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport math\nimport os\nimport sys\nfrom typing import Dict, List, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom training.restoration_dataset import DocumentRestorationDataset\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger("train_restoration")\n\n\n# ==============================================================================\n# ARCHITECTURAL BLOCKS (U-Net v5)\n# ==============================================================================\n\nclass ResidualBlock(nn.Module):\n    """Residual block with two 3x3 convolutions and GroupNorm (8 groups)."""\n\n    def __init__(self, channels: int):\n        super().__init__()\n        num_groups = min(8, channels)\n        self.conv = nn.Sequential(\n            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),\n            nn.GroupNorm(num_groups=num_groups, num_channels=channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),\n            nn.GroupNorm(num_groups=num_groups, num_channels=channels),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return x + self.conv(x)\n\n\nclass AttentionGate(nn.Module):\n    """\n    Attention Gate (Oktay et al., 2018):\n    Filters background noise and back-side bleed-through before skip connections.\n    F_g: channels of gating signal from decoder\n    F_l: channels of skip connection from encoder\n    F_int: intermediate channel dimension\n    """\n\n    def __init__(self, F_g: int, F_l: int, F_int: int):\n        super().__init__()\n        num_groups = min(8, F_int)\n        self.W_g = nn.Sequential(\n            nn.Conv2d(F_g, F_int, kernel_size=1, bias=False),\n            nn.GroupNorm(num_groups=num_groups, num_channels=F_int),\n        )\n        self.W_x = nn.Sequential(\n            nn.Conv2d(F_l, F_int, kernel_size=1, bias=False),\n            nn.GroupNorm(num_groups=num_groups, num_channels=F_int),\n        )\n        self.psi = nn.Sequential(\n            nn.Conv2d(F_int, 1, kernel_size=1, bias=True),\n            nn.Sigmoid(),\n        )\n        self.relu = nn.ReLU(inplace=True)\n\n    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:\n        g1 = self.W_g(g)\n        x1 = self.W_x(x)\n        psi = self.relu(g1 + x1)\n        alpha = self.psi(psi)\n        return x * alpha\n\n\nclass ASPP(nn.Module):\n    """\n    Atrous Spatial Pyramid Pooling for multi-receptive field context in the bottleneck.\n    Dilations: 1, 2, 4, 8 + Global Average Pooling.\n    """\n\n    def __init__(self, in_channels: int = 512, out_channels: int = 512):\n        super().__init__()\n        branch_ch = 128\n        self.conv1 = nn.Sequential(\n            nn.Conv2d(in_channels, branch_ch, kernel_size=1, bias=False),\n            nn.GroupNorm(8, branch_ch),\n            nn.ReLU(inplace=True),\n        )\n        self.conv2 = nn.Sequential(\n            nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=1, dilation=1, bias=False),\n            nn.GroupNorm(8, branch_ch),\n            nn.ReLU(inplace=True),\n        )\n        self.conv3 = nn.Sequential(\n            nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=2, dilation=2, bias=False),\n            nn.GroupNorm(8, branch_ch),\n            nn.ReLU(inplace=True),\n        )\n        self.conv4 = nn.Sequential(\n            nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=4, dilation=4, bias=False),\n            nn.GroupNorm(8, branch_ch),\n            nn.ReLU(inplace=True),\n        )\n        self.conv5 = nn.Sequential(\n            nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=8, dilation=8, bias=False),\n            nn.GroupNorm(8, branch_ch),\n            nn.ReLU(inplace=True),\n        )\n        self.global_pool = nn.Sequential(\n            nn.AdaptiveAvgPool2d((1, 1)),\n            nn.Conv2d(in_channels, branch_ch, kernel_size=1, bias=False),\n            nn.GroupNorm(8, branch_ch),\n            nn.ReLU(inplace=True),\n        )\n        total_in = branch_ch * 6\n        self.project = nn.Sequential(\n            nn.Conv2d(total_in, out_channels, kernel_size=1, bias=False),\n            nn.GroupNorm(8, out_channels),\n            nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        h, w = x.shape[2], x.shape[3]\n        f1 = self.conv1(x)\n        f2 = self.conv2(x)\n        f3 = self.conv3(x)\n        f4 = self.conv4(x)\n        f5 = self.conv5(x)\n        gp = self.global_pool(x)\n        gp_up = F.interpolate(gp, size=(h, w), mode="bilinear", align_corners=False)\n        cat = torch.cat([f1, f2, f3, f4, f5, gp_up], dim=1)\n        return self.project(cat)\n\n\n# ==============================================================================\n# LOSS FUNCTIONS\n# ==============================================================================\n\nclass SobelLoss(nn.Module):\n    """Multi-Scale Sobel Gradient Loss to enforce sharp text edge reconstruction across scales."""\n\n    def __init__(self):\n        super().__init__()\n        sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], dtype=torch.float32).view(1, 1, 3, 3)\n        sobel_y = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]], dtype=torch.float32).view(1, 1, 3, 3)\n        self.register_buffer("sobel_x", sobel_x)\n        self.register_buffer("sobel_y", sobel_y)\n\n    def _grad_l1(self, p: torch.Tensor, t: torch.Tensor) -> torch.Tensor:\n        p_gx = F.conv2d(p, self.sobel_x, padding=1)\n        p_gy = F.conv2d(p, self.sobel_y, padding=1)\n        t_gx = F.conv2d(t, self.sobel_x, padding=1)\n        t_gy = F.conv2d(t, self.sobel_y, padding=1)\n        return F.l1_loss(p_gx, t_gx) + F.l1_loss(p_gy, t_gy)\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if pred.shape[1] == 3:\n            pred_g = 0.299 * pred[:, 0:1] + 0.587 * pred[:, 1:2] + 0.114 * pred[:, 2:3]\n            target_g = 0.299 * target[:, 0:1] + 0.587 * target[:, 1:2] + 0.114 * target[:, 2:3]\n        else:\n            pred_g = pred\n            target_g = target\n\n        loss_scale1 = self._grad_l1(pred_g, target_g)\n        if pred_g.shape[-2] >= 8 and pred_g.shape[-1] >= 8:\n            pred_half = F.avg_pool2d(pred_g, kernel_size=2, stride=2)\n            target_half = F.avg_pool2d(target_g, kernel_size=2, stride=2)\n            loss_scale2 = self._grad_l1(pred_half, target_half)\n            return loss_scale1 + 0.5 * loss_scale2\n        return loss_scale1\n\n\nclass SSIMLoss(nn.Module):\n    """Differentiable SSIM Loss (1 - SSIM) using an 11x11 Gaussian window."""\n\n    def __init__(self, window_size: int = 11, sigma: float = 1.5):\n        super().__init__()\n        self.window_size = window_size\n        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2\n        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))\n        g = g / g.sum()\n        window_2d = (g.unsqueeze(1) @ g.unsqueeze(0)).unsqueeze(0).unsqueeze(0)\n        self.register_buffer("window", window_2d)\n        self.c1 = 0.01 ** 2\n        self.c2 = 0.03 ** 2\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n        channels = pred.shape[1]\n        win = self.window.expand(channels, 1, self.window_size, self.window_size)\n        pad = self.window_size // 2\n\n        mu1 = F.conv2d(pred, win, padding=pad, groups=channels)\n        mu2 = F.conv2d(target, win, padding=pad, groups=channels)\n        mu1_sq = mu1 * mu1\n        mu2_sq = mu2 * mu2\n        mu1_mu2 = mu1 * mu2\n\n        sigma1_sq = F.conv2d(pred * pred, win, padding=pad, groups=channels) - mu1_sq\n        sigma2_sq = F.conv2d(target * target, win, padding=pad, groups=channels) - mu2_sq\n        sigma12 = F.conv2d(pred * target, win, padding=pad, groups=channels) - mu1_mu2\n\n        ssim_map = ((2.0 * mu1_mu2 + self.c1) * (2.0 * sigma12 + self.c2)) / (\n            (mu1_sq + mu2_sq + self.c1) * (sigma1_sq + sigma2_sq + self.c2) + 1e-8\n        )\n        return torch.clamp(1.0 - ssim_map.mean(), min=0.0)\n\n\nclass AsymmetricStrokeDarkeningLoss(nn.Module):\n    """\n    Asymmetric stroke darkening penalty:\n    Penalizes faded or gray predictions (pred > target) specifically on text stroke pixels\n    where target < target_thresh (default 0.70). Does NOT penalize dark strokes.\n    """\n\n    def __init__(self, target_thresh: float = 0.70):\n        super().__init__()\n        self.target_thresh = target_thresh\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n        stroke_mask = (target < self.target_thresh).float()\n        faded_penalty = torch.mean(stroke_mask * (F.relu(pred - target) ** 2))\n        return faded_penalty\n\n\nclass BleedThroughSuppressionLoss(nn.Module):\n    """\n    Penalizes residual gray predictions (pred < pred_floor) in regions where target > target_thresh\n    (pure white background or back-side bleed-through/watermark regions that must be erased).\n    """\n\n    def __init__(self, target_thresh: float = 0.95, pred_floor: float = 0.92):\n        super().__init__()\n        self.target_thresh = target_thresh\n        self.pred_floor = pred_floor\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n        bg_or_bleed_mask = (target > self.target_thresh).float()\n        below_floor = F.relu(self.pred_floor - pred)\n        white_gap = F.relu(target - pred)\n        return torch.mean(bg_or_bleed_mask * (10.0 * (below_floor ** 2) + 0.5 * white_gap))\n\n\nclass FFTLoss(nn.Module):\n    """Frequency-domain L1 loss using 2D Real FFT to preserve fine text texture and suppress noise grids."""\n\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred_fft = torch.fft.rfft2(pred.float(), norm="ortho")\n        target_fft = torch.fft.rfft2(target.float(), norm="ortho")\n        pred_mag = torch.abs(pred_fft)\n        target_mag = torch.abs(target_fft)\n        return F.l1_loss(pred_mag, target_mag)\n\n\nclass DocumentCleanLoss(nn.Module):\n    """\n    Comprehensive Multi-Task Loss for Document Restoration (v5):\n    - Weighted Charbonnier Loss (1.0)\n    - Multi-Scale Sobel Gradient Edge Loss (0.8)\n    - Structural Similarity SSIM Loss (0.2)\n    - Asymmetric Stroke Darkening Loss (0.25)\n    - Bleed-Through Suppression Loss (0.15)\n    - FFT Frequency L1 Loss (0.05)\n    """\n\n    def __init__(self):\n        super().__init__()\n        self.sobel = SobelLoss()\n        self.ssim = SSIMLoss()\n        self.stroke_darkening = AsymmetricStrokeDarkeningLoss(target_thresh=0.70)\n        self.bleed_suppression = BleedThroughSuppressionLoss(target_thresh=0.95, pred_floor=0.92)\n        self.fft_loss = FFTLoss()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n\n        # Weighted Charbonnier prioritizing stroke reconstruction\n        stroke_weight = 1.0 + 6.0 * ((1.0 - target) ** 2)\n        charbonnier = torch.sqrt((pred - target) ** 2 + 1e-6)\n        loss_charbonnier = torch.mean(stroke_weight * charbonnier)\n\n        loss_sobel = self.sobel(pred, target)\n        loss_ssim = self.ssim(pred, target)\n        loss_darkening = self.stroke_darkening(pred, target)\n        loss_bleed = self.bleed_suppression(pred, target)\n        loss_fft = self.fft_loss(pred, target)\n\n        total_loss = (\n            1.0 * loss_charbonnier\n            + 0.8 * loss_sobel\n            + 0.2 * loss_ssim\n            + 0.25 * loss_darkening\n            + 0.15 * loss_bleed\n            + 0.05 * loss_fft\n        )\n        return total_loss\n\n\n# ==============================================================================\n# MODEL ARCHITECTURE: DocumentRestorationUNet (v5)\n# ==============================================================================\n\nclass DocumentRestorationUNet(nn.Module):\n    """\n    DocumentRestorationUNet v5:\n    - Multi-Scale Robust Input (3 channels): [x, x_norm_robust_31, x_norm_robust_101]\n    - GroupNorm(8) throughout internal layers (no batch-size dependence or tile illumination shifts)\n    - Bilinear Upsampling + Conv3x3 (no checkerboard artifacts)\n    - Bottleneck: 1 ResidualBlock(512) + ASPP(512)\n    - Selective Skip Connections: AttentionGate on levels 2 & 3; direct skip on level 1\n    - Logit-Space Residual Output Head: delta = out_conv(d1), y = sigmoid(logit(x_norm_31) + delta)\n    """\n\n    def __init__(self, in_channels: int = 1, out_channels: int = 1):\n        super().__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n\n        # 3-channel input: [x, x_norm_robust_31, x_norm_robust_101]\n        enc_in_channels = in_channels * 3\n\n        # Encoder level 1\n        self.enc1 = nn.Sequential(\n            nn.Conv2d(enc_in_channels, 64, kernel_size=3, padding=1),\n            ResidualBlock(64),\n        )\n        self.down1 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)\n\n        # Encoder level 2\n        self.enc2 = nn.Sequential(\n            ResidualBlock(128),\n            ResidualBlock(128),\n        )\n        self.down2 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)\n\n        # Encoder level 3\n        self.enc3 = nn.Sequential(\n            ResidualBlock(256),\n            ResidualBlock(256),\n        )\n        self.down3 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)\n\n        # Bottleneck: 1 ResidualBlock + ASPP\n        self.bottleneck = nn.Sequential(\n            ResidualBlock(512),\n            ASPP(512, 512),\n        )\n\n        # Decoder level 3 (Attention Gate on e3)\n        self.up3 = nn.Sequential(\n            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),\n            nn.Conv2d(512, 256, kernel_size=3, padding=1, bias=False),\n        )\n        self.att3 = AttentionGate(F_g=256, F_l=256, F_int=128)\n        self.dec3 = nn.Sequential(\n            nn.Conv2d(256 + 256, 256, kernel_size=1),\n            ResidualBlock(256),\n            ResidualBlock(256),\n        )\n\n        # Decoder level 2 (Attention Gate on e2)\n        self.up2 = nn.Sequential(\n            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),\n            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),\n        )\n        self.att2 = AttentionGate(F_g=128, F_l=128, F_int=64)\n        self.dec2 = nn.Sequential(\n            nn.Conv2d(128 + 128, 128, kernel_size=1),\n            ResidualBlock(128),\n            ResidualBlock(128),\n        )\n\n        # Decoder level 1 (Direct skip connection on e1 to preserve 100% fine accents)\n        self.up1 = nn.Sequential(\n            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),\n            nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False),\n        )\n        self.dec1 = nn.Sequential(\n            nn.Conv2d(64 + 64, 64, kernel_size=1),\n            ResidualBlock(64),\n        )\n\n        # Logit-space residual head\n        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=3, padding=1)\n        nn.init.zeros_(self.out_conv.weight)\n        if self.out_conv.bias is not None:\n            nn.init.zeros_(self.out_conv.bias)\n\n    @staticmethod\n    def _median_filter_3x3(x: torch.Tensor) -> torch.Tensor:\n        b, c, h, w = x.shape\n        x_pad = F.pad(x, (1, 1, 1, 1), mode="reflect")\n        patches = F.unfold(x_pad, kernel_size=3, stride=1)\n        med = patches.median(dim=1).values.view(b, c, h, w)\n        return med\n\n    @staticmethod\n    def _get_norm_robust(x_med: torch.Tensor, x_raw: torch.Tensor, k: int) -> torch.Tensor:\n        pad = k // 2\n        x_pad = F.pad(x_med, (pad, pad, pad, pad), mode="reflect")\n        bg_max = F.max_pool2d(x_pad, kernel_size=k, stride=1, padding=0)\n        bg_pad = F.pad(bg_max, (pad, pad, pad, pad), mode="reflect")\n        bg_smooth = F.avg_pool2d(bg_pad, kernel_size=k, stride=1, padding=0)\n        norm = torch.clamp(x_raw / (bg_smooth + 1e-4), 0.0, 1.0)\n        return norm\n\n    def _load_from_state_dict(\n        self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs\n    ):\n        """Allow seamless loading of legacy v4 checkpoints if encountered."""\n        w_key = prefix + "enc1.0.weight"\n        if w_key in state_dict:\n            w_val = state_dict[w_key]\n            expected_shape = self.enc1[0].weight.shape\n            if w_val.ndim == 4 and w_val.shape[1] in (1, 2) and expected_shape[1] == 3:\n                channels_needed = 3 - w_val.shape[1]\n                zeros = torch.zeros(w_val.shape[0], channels_needed, w_val.shape[2], w_val.shape[3], device=w_val.device, dtype=w_val.dtype)\n                state_dict[w_key] = torch.cat([w_val, zeros], dim=1)\n\n        super()._load_from_state_dict(\n            state_dict, prefix, local_metadata, False, missing_keys, unexpected_keys, error_msgs\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # Pre-filtering with 3x3 median filter\n        x_med = self._median_filter_3x3(x)\n\n        # 2 scales: k=31 and k=101\n        x_norm_31 = self._get_norm_robust(x_med, x, 31)\n        x_norm_101 = self._get_norm_robust(x_med, x, 101)\n\n        x_in = torch.cat([x, x_norm_31, x_norm_101], dim=1)  # [B, 3, H, W]\n\n        # Encoder\n        e1 = self.enc1(x_in)    # [B, 64, H, W]\n        d1 = self.down1(e1)     # [B, 128, H/2, W/2]\n\n        e2 = self.enc2(d1)      # [B, 128, H/2, W/2]\n        d2 = self.down2(e2)     # [B, 256, H/4, W/4]\n\n        e3 = self.enc3(d2)      # [B, 256, H/4, W/4]\n        d3 = self.down3(e3)     # [B, 512, H/8, W/8]\n\n        # Bottleneck (ResBlock + ASPP)\n        b = self.bottleneck(d3) # [B, 512, H/8, W/8]\n\n        # Decoder level 3 (Attention Gate on e3)\n        u3 = self.up3(b)\n        e3_att = self.att3(g=u3, x=e3)\n        c3 = torch.cat([u3, e3_att], dim=1)\n        d_out3 = self.dec3(c3)\n\n        # Decoder level 2 (Attention Gate on e2)\n        u2 = self.up2(d_out3)\n        e2_att = self.att2(g=u2, x=e2)\n        c2 = torch.cat([u2, e2_att], dim=1)\n        d_out2 = self.dec2(c2)\n\n        # Decoder level 1 (Direct skip on e1)\n        u1 = self.up1(d_out2)\n        c1 = torch.cat([u1, e1], dim=1)\n        d_out1 = self.dec1(c1)\n\n        # Logit-space residual head\n        delta = self.out_conv(d_out1)\n        base_clamped = torch.clamp(x_norm_31, 1e-4, 1.0 - 1e-4)\n        base_logit = torch.logit(base_clamped)\n        y_pred = torch.sigmoid(base_logit + delta)\n        return y_pred\n\n\n# ==============================================================================\n# EVALUATION METRICS\n# ==============================================================================\n\ndef calculate_psnr(img1: torch.Tensor, img2: torch.Tensor) -> float:\n    """Calculate PSNR between two tensors with values in [0, 1]."""\n    mse = torch.mean((img1 - img2) ** 2).item()\n    if mse == 0:\n        return 100.0\n    return 20.0 * math.log10(1.0 / math.sqrt(mse))\n\n\n# ==============================================================================\n# TRAINING PIPELINE\n# ==============================================================================\n\ndef train_restoration_model(\n    train_csv: str = "datasets/metadata/train.csv",\n    val_csv: str = "datasets/metadata/val.csv",\n    output_dir: str = "models",\n    epochs: int = 40,\n    batch_size: int = 4,\n    lr: float = 2e-4,\n    image_size: int = 512,\n    manifest: Optional[str] = None,\n    dataset_root: Optional[str] = None,\n    resume_weights: Optional[str] = None,\n):\n    """\n    Train or Fine-Tune Document Restoration UNet v5 with validation and best checkpoint saving.\n    """\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    logger.info(f"Starting Document Restoration UNet v5 Training on device: {device}")\n\n    img_size = (image_size, image_size)\n    if manifest:\n        train_csv = val_csv = manifest\n        train_dataset = DocumentRestorationDataset(manifest, image_size=img_size, split="train", is_train=True, root_dir=dataset_root)\n        val_dataset = DocumentRestorationDataset(manifest, image_size=img_size, split="val", is_train=False, root_dir=dataset_root)\n    else:\n        train_dataset = DocumentRestorationDataset(metadata_csv=train_csv, image_size=img_size, split="train", is_train=True)\n        val_dataset = DocumentRestorationDataset(metadata_csv=val_csv, image_size=img_size, split="val", is_train=False)\n\n    if len(train_dataset) == 0:\n        logger.error(f"No training pairs found in \'{train_csv}\'. Training aborted.")\n        raise ValueError(f"No training pairs found in \'{train_csv}\'. Training aborted.")\n\n    logger.info(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")\n\n    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)\n    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0) if len(val_dataset) > 0 else []\n\n    # Model\n    model = DocumentRestorationUNet().to(device)\n    total_params = sum(p.numel() for p in model.parameters())\n    logger.info(f"U-Net v5 Model parameters: {total_params:,}")\n\n    if resume_weights and os.path.exists(resume_weights):\n        try:\n            state_dict = torch.load(resume_weights, map_location=device, weights_only=True)\n            model.load_state_dict(state_dict)\n            logger.info(f"Loaded pretrained weights from \'{resume_weights}\'")\n        except Exception as e:\n            logger.warning(f"Could not load checkpoint \'{resume_weights}\' directly ({e}). Initializing fresh v5 weights.")\n\n    criterion = DocumentCleanLoss().to(device)\n    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)\n\n    os.makedirs(output_dir, exist_ok=True)\n    best_model_path = os.path.join(output_dir, "restoration_best.pth")\n    last_model_path = os.path.join(output_dir, "restoration_last.pth")\n\n    best_val_psnr = -float("inf")\n    training_log = []\n\n    for epoch in range(1, epochs + 1):\n        # TRAIN\n        model.train()\n        train_loss = 0.0\n        train_psnr = 0.0\n        train_steps = 0\n\n        for deg, clean in train_loader:\n            deg, clean = deg.to(device), clean.to(device)\n            optimizer.zero_grad()\n\n            pred = model(deg)\n            loss = criterion(pred, clean)\n            loss.backward()\n            optimizer.step()\n\n            train_loss += loss.item()\n            with torch.no_grad():\n                train_psnr += calculate_psnr(pred, clean)\n            train_steps += 1\n\n        scheduler.step()\n        avg_train_loss = train_loss / max(train_steps, 1)\n        avg_train_psnr = train_psnr / max(train_steps, 1)\n\n        # VALIDATION\n        avg_val_loss = 0.0\n        avg_val_psnr = 0.0\n\n        if len(val_dataset) > 0:\n            model.eval()\n            val_loss = 0.0\n            val_psnr = 0.0\n            val_steps = 0\n\n            with torch.no_grad():\n                for deg, clean in val_loader:\n                    deg, clean = deg.to(device), clean.to(device)\n                    pred = model(deg)\n                    loss = criterion(pred, clean)\n                    val_loss += loss.item()\n                    val_psnr += calculate_psnr(pred, clean)\n                    val_steps += 1\n\n            avg_val_loss = val_loss / max(val_steps, 1)\n            avg_val_psnr = val_psnr / max(val_steps, 1)\n\n        epoch_info = {\n            "epoch": epoch,\n            "train_loss": round(avg_train_loss, 6),\n            "train_psnr": round(avg_train_psnr, 2),\n            "val_loss": round(avg_val_loss, 6),\n            "val_psnr": round(avg_val_psnr, 2),\n            "lr": round(scheduler.get_last_lr()[0], 7),\n        }\n        training_log.append(epoch_info)\n\n        logger.info(\n            f"Epoch [{epoch}/{epochs}] "\n            f"Train Loss: {avg_train_loss:.4f} | Train PSNR: {avg_train_psnr:.2f} dB | "\n            f"Val Loss: {avg_val_loss:.4f} | Val PSNR: {avg_val_psnr:.2f} dB | "\n            f"LR: {scheduler.get_last_lr()[0]:.2e}"\n        )\n\n        torch.save(model.state_dict(), last_model_path)\n\n        if avg_val_psnr > best_val_psnr and len(val_dataset) > 0:\n            best_val_psnr = avg_val_psnr\n            torch.save(model.state_dict(), best_model_path)\n            logger.info(f"  -> New best model saved (Val PSNR: {best_val_psnr:.2f} dB)")\n\n    if len(val_dataset) == 0:\n        torch.save(model.state_dict(), best_model_path)\n        logger.info("No validation data. Saved last model as best.")\n\n    log_path = os.path.join(output_dir, "training_log.json")\n    val_psnr_record = round(best_val_psnr, 2) if best_val_psnr != -float("inf") else None\n    with open(log_path, "w", encoding="utf-8") as f:\n        json.dump({\n            "device": str(device),\n            "epochs": epochs,\n            "batch_size": batch_size,\n            "learning_rate": lr,\n            "image_size": image_size,\n            "train_samples": len(train_dataset),\n            "val_samples": len(val_dataset),\n            "best_val_psnr": val_psnr_record,\n            "log": training_log,\n        }, f, indent=2)\n\n    logger.info(f"Training complete. Best Val PSNR: {val_psnr_record if val_psnr_record is not None else \'N/A\'} dB")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train Document Image Restoration UNet v5")\n    parser.add_argument("--train-csv", default="datasets/metadata/train.csv")\n    parser.add_argument("--val-csv", default="datasets/metadata/val.csv")\n    parser.add_argument("--output-dir", default="models")\n    parser.add_argument("--epochs", type=int, default=40)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--lr", type=float, default=2e-4)\n    parser.add_argument("--image-size", type=int, default=512)\n    parser.add_argument("--manifest", default=None)\n    parser.add_argument("--dataset-root", default=None)\n    parser.add_argument("--resume-weights", default=None)\n    args = parser.parse_args()\n\n    train_restoration_model(\n        train_csv=args.train_csv,\n        val_csv=args.val_csv,\n        output_dir=args.output_dir,\n        epochs=args.epochs,\n        batch_size=args.batch_size,\n        lr=args.lr,\n        image_size=args.image_size,\n        manifest=args.manifest,\n        dataset_root=args.dataset_root,\n        resume_weights=args.resume_weights,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

EVALUATE_RESTORATION_CODE = 'import os\nimport sys\nimport argparse\nimport csv\nimport cv2\nimport numpy as np\nimport torch\nimport math\nimport json\nfrom PIL import Image\n\n# Add parent directory to path to import training modules\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \'..\')))\nfrom training.train_restoration import DocumentRestorationUNet\nfrom training.restoration_dataset import (\n    load_image,\n    save_image,\n    unsharp_mask,\n    resize_with_padding,\n    remove_padding,\n    normalize_clean_target,\n    clean_target_normalization,\n    infer_single_image_tiled,\n    GEOMETRIC_DISTORTION_KEYWORDS,\n)\n\n\ndef calculate_psnr(img1: np.ndarray, img2: np.ndarray) -> float:\n    """Calculate PSNR between two uint8 images."""\n    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)\n    if mse == 0:\n        return 100.0\n    return 20 * math.log10(255.0 / math.sqrt(mse))\n\n\ndef calculate_ssim(img1: np.ndarray, img2: np.ndarray) -> float:\n    """Calculate SSIM between two uint8 images."""\n    c1 = (0.01 * 255) ** 2\n    c2 = (0.03 * 255) ** 2\n    img1 = img1.astype(np.float64)\n    img2 = img2.astype(np.float64)\n    mu1 = cv2.GaussianBlur(img1, (11, 11), 1.5)\n    mu2 = cv2.GaussianBlur(img2, (11, 11), 1.5)\n    mu1_sq = mu1 ** 2\n    mu2_sq = mu2 ** 2\n    mu1_mu2 = mu1 * mu2\n    sigma1_sq = cv2.GaussianBlur(img1 ** 2, (11, 11), 1.5) - mu1_sq\n    sigma2_sq = cv2.GaussianBlur(img2 ** 2, (11, 11), 1.5) - mu2_sq\n    sigma12 = cv2.GaussianBlur(img1 * img2, (11, 11), 1.5) - mu1_mu2\n    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / (\n        (mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2)\n    )\n    return float(np.mean(ssim_map))\n\n\ndef _load_pairs_from_csv(csv_path: str, default_split: str = "test"):\n    """Load valid image pairs and unpaired blind evaluation entries from a split CSV (test.csv or val.csv)."""\n    pairs = []\n    unpaired = []\n    if not csv_path or not os.path.exists(csv_path):\n        return pairs, unpaired\n    with open(csv_path, \'r\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n            raw_in = (row.get(\'input\') or \'\').strip()\n            raw_tgt = (row.get(\'target\') or \'\').strip()\n            in_p = os.path.normpath(raw_in) if raw_in else \'\'\n            tgt_p = os.path.normpath(raw_tgt) if raw_tgt else \'\'\n            deg_str = (row.get(\'degradation\') or row.get(\'blur_level\') or \'\').strip().lower()\n            in_stem = os.path.splitext(os.path.basename(in_p))[0].lower() if in_p else \'\'\n\n            is_blind = (\n                in_stem in (\'real_11\', \'real_12\')\n                or deg_str in (\'unpaired_real\', \'blind_real\')\n                or (raw_tgt == \'\' and bool(raw_in))\n            )\n            if is_blind:\n                if in_p and os.path.exists(in_p):\n                    row_copy = dict(row)\n                    row_copy[\'input\'] = in_p\n                    row_copy[\'target\'] = \'\'\n                    row_copy[\'split\'] = row.get(\'split\', default_split) or default_split\n                    row_copy[\'is_unpaired\'] = True\n                    unpaired.append(row_copy)\n                continue\n\n            if in_p and tgt_p and os.path.exists(in_p) and os.path.exists(tgt_p):\n                row_copy = dict(row)\n                row_copy[\'input\'] = in_p\n                row_copy[\'target\'] = tgt_p\n                row_copy[\'split\'] = row.get(\'split\', default_split) or default_split\n                row_copy[\'is_unpaired\'] = False\n                pairs.append(row_copy)\n    return pairs, unpaired\n\n\ndef _collect_blind_real_samples(existing_unpaired):\n    """Ensure datasets/processed/test/input/real_*.jpg and datasets/processed/val/input/real_*.jpg are included."""\n    seen_paths = {os.path.abspath(item[\'input\']) for item in existing_unpaired if item.get(\'input\')}\n    blind_list = list(existing_unpaired)\n\n    candidates = [\n        (\'test\', \'datasets/processed/test/input/real_11.jpg\'),\n        (\'val\', \'datasets/processed/val/input/real_12.jpg\'),\n    ]\n    for split_name, folder in [(\'test\', \'datasets/processed/test/input\'), (\'val\', \'datasets/processed/val/input\')]:\n        if os.path.exists(folder):\n            for fn in sorted(os.listdir(folder)):\n                if fn.lower().startswith(\'real_\') and os.path.splitext(fn)[1].lower() in (\'.jpg\', \'.jpeg\', \'.png\', \'.webp\'):\n                    candidates.append((split_name, os.path.join(folder, fn)))\n\n    for split_name, rel_p in candidates:\n        norm_p = os.path.normpath(rel_p)\n        if os.path.exists(norm_p):\n            abs_p = os.path.abspath(norm_p)\n            if abs_p not in seen_paths:\n                seen_paths.add(abs_p)\n                stem = os.path.splitext(os.path.basename(norm_p))[0]\n                blind_list.append({\n                    \'input\': norm_p,\n                    \'target\': \'\',\n                    \'source_id\': stem,\n                    \'degradation\': \'unpaired_real\',\n                    \'blur_level\': f\'real_blind_{split_name}\',\n                    \'split\': split_name,\n                    \'is_unpaired\': True,\n                })\n    return blind_list\n\n\ndef create_two_column_blind_comparison(raw_bgr: np.ndarray, pure_unet_gray: np.ndarray, target_h: int = 680) -> np.ndarray:\n    """\n    Create an honest 2-column comparison sheet for Unpaired Real Evaluation:\n    [1. Anh Real Goc (Input duy nhat) | 2. Ket qua 100% Pure U-Net (Output tu AI)]\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    if pure_unet_gray.ndim == 2:\n        unet_bgr = cv2.cvtColor(pure_unet_gray, cv2.COLOR_GRAY2BGR)\n    else:\n        unet_bgr = pure_unet_gray.copy()\n\n    def _resize_col(img: np.ndarray, th: int) -> np.ndarray:\n        hi, wi = img.shape[:2]\n        scale = th / float(max(hi, 1))\n        nw = max(240, int(wi * scale))\n        return cv2.resize(img, (nw, th), interpolation=cv2.INTER_AREA)\n\n    def _add_header(img: np.ndarray, title: str, color: tuple) -> np.ndarray:\n        header_h = 46\n        canvas = np.full((img.shape[0] + header_h, img.shape[1], 3), 245, dtype=np.uint8)\n        canvas[header_h:, :] = img\n        cv2.putText(canvas, title, (12, 31), cv2.FONT_HERSHEY_SIMPLEX, 0.62, color, 2, cv2.LINE_AA)\n        return canvas\n\n    col1 = _add_header(\n        _resize_col(raw_bgr, target_h),\n        "1. Anh Real Goc (Input duy nhat)",\n        (0, 0, 210),\n    )\n    col2 = _add_header(\n        _resize_col(unet_bgr, target_h),\n        "2. Ket qua 100% Pure U-Net (Output tu AI)",\n        (180, 60, 0),\n    )\n    divider = np.full((col1.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col1, divider, col2])\n\n\ndef _summarize_group(items):\n    """Compute average PSNR/SSIM summary for a subset of evaluated samples."""\n    if not items:\n        return {\n            \'count\': 0,\n            \'restored_psnr\': 0.0,\n            \'blur_psnr\': 0.0,\n            \'psnr_improvement\': 0.0,\n            \'restored_ssim\': 0.0,\n            \'blur_ssim\': 0.0,\n            \'ssim_improvement\': 0.0,\n        }\n    r_psnr = float(np.mean([m[\'restored_psnr\'] for m in items]))\n    b_psnr = float(np.mean([m[\'blur_psnr\'] for m in items]))\n    r_ssim = float(np.mean([m[\'restored_ssim\'] for m in items]))\n    b_ssim = float(np.mean([m[\'blur_ssim\'] for m in items]))\n    return {\n        \'count\': len(items),\n        \'restored_psnr\': round(r_psnr, 2),\n        \'blur_psnr\': round(b_psnr, 2),\n        \'psnr_improvement\': round(r_psnr - b_psnr, 2),\n        \'restored_ssim\': round(r_ssim, 4),\n        \'blur_ssim\': round(b_ssim, 4),\n        \'ssim_improvement\': round(r_ssim - b_ssim, 4),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Evaluate Document Restoration AI Model")\n    parser.add_argument(\'--model\', type=str, default=\'models/restoration_best.pth\',\n                        help=\'Path to model checkpoint\')\n    parser.add_argument(\'--test-csv\', type=str, default=\'datasets/metadata/test.csv\',\n                        help=\'Path to test split CSV\')\n    parser.add_argument(\'--val-csv\', type=str, default=\'datasets/metadata/val.csv\',\n                        help=\'Path to val split CSV\')\n    parser.add_argument(\'--include-val\', action=\'store_true\', default=True,\n                        help=\'Also export comparison sheets for val.csv\')\n    parser.add_argument(\'--no-val\', action=\'store_false\', dest=\'include_val\',\n                        help=\'Evaluate only test.csv without val.csv\')\n    parser.add_argument(\'--image-size\', type=int, default=512,\n                        help=\'Target model tile size for sliding window inference\')\n    parser.add_argument(\'--overlap\', type=int, default=128,\n                        help=\'Overlap between sliding window tiles\')\n    parser.add_argument(\'--blind-out-dir\', type=str, default=\'outputs/restoration/real_blind_test\',\n                        help=\'Output directory for 100% Pure U-Net blind test images (real_11.jpg & real_12.jpg)\')\n    args = parser.parse_args()\n\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(f"Device: {device}")\n\n    # Load Model\n    model = DocumentRestorationUNet().to(device)\n\n    if os.path.exists(args.model):\n        model.load_state_dict(torch.load(args.model, map_location=device, weights_only=True))\n        print(f"Loaded model from {args.model}")\n    else:\n        print(f"ERROR: Model not found at {args.model}.")\n        return\n\n    model.eval()\n\n    output_dir = \'outputs/restoration/test\'\n    comp_dir = \'outputs/restoration/comparison\'\n    blind_dir = args.blind_out_dir\n    os.makedirs(output_dir, exist_ok=True)\n    os.makedirs(comp_dir, exist_ok=True)\n    os.makedirs(blind_dir, exist_ok=True)\n\n    # Remove old comparison images so only current evaluation results exist\n    for old_f in os.listdir(comp_dir):\n        if old_f.endswith("_comparison.png"):\n            try:\n                os.remove(os.path.join(comp_dir, old_f))\n            except OSError:\n                pass\n\n    # Read test CSV and val CSV\n    test_pairs, test_unpaired = _load_pairs_from_csv(args.test_csv, default_split="test")\n    val_pairs, val_unpaired = _load_pairs_from_csv(args.val_csv, default_split="val") if args.include_val else ([], [])\n\n    eval_pairs = test_pairs + val_pairs\n    blind_samples = _collect_blind_real_samples(test_unpaired + val_unpaired)\n\n    if len(eval_pairs) == 0:\n        manifest_p = "datasets/metadata/dataset_manifest.csv"\n        if os.path.exists(manifest_p):\n            print(f"No valid files in \'{args.test_csv}\'. Loading evaluation pairs from \'{manifest_p}\'...")\n            with open(manifest_p, \'r\', encoding=\'utf-8-sig\') as f:\n                reader = csv.DictReader(f)\n                for r in reader:\n                    if r.get("split", "").strip() not in ("test", "val"):\n                        continue\n                    raw_in = r.get("input_path", "").strip()\n                    raw_tgt = r.get("target_path", "").strip()\n                    if not raw_in or not raw_tgt:\n                        continue\n                    in_p = os.path.normpath(os.path.join("datasets", raw_in))\n                    tgt_p = os.path.normpath(os.path.join("datasets", raw_tgt))\n                    if os.path.exists(in_p) and os.path.exists(tgt_p):\n                        eval_pairs.append({\n                            \'input\': in_p,\n                            \'target\': tgt_p,\n                            \'source_id\': r.get(\'original_id\', \'doc\'),\n                            \'blur_level\': r.get(\'degradation_type\', \'degraded\') + \'_\' + r.get(\'variant_id\', \'01\'),\n                            \'split\': r.get(\'split\', \'test\'),\n                        })\n        elif len(blind_samples) == 0:\n            print("ERROR: No valid test pairs found.")\n            return\n\n    print(\n        f"Total paired samples to evaluate: {len(eval_pairs)} (test={len(test_pairs)}, val={len(val_pairs)}) | "\n        f"Unpaired Blind Real samples: {len(blind_samples)}"\n    )\n    all_metrics = []\n\n    for pair in eval_pairs:\n        blur_path = os.path.normpath(pair[\'input\'].strip())\n        orig_path = os.path.normpath(pair[\'target\'].strip())\n        source_id = pair.get(\'source_id\') or pair.get(\'original_id\') or \'unknown\'\n        blur_level = pair.get(\'blur_level\') or pair.get(\'degradation\') or os.path.splitext(os.path.basename(blur_path))[0]\n        split_tag = pair.get(\'split\', \'test\')\n\n        blur_img = load_image(blur_path)\n        orig_img = load_image(orig_path)\n        if blur_img is None or orig_img is None:\n            print(f"  SKIP {source_id} ({blur_level}): failed to load images")\n            continue\n\n        # Convert to Grayscale and apply synchronized clean_target_normalization on Ground Truth\n        blur_gray = cv2.cvtColor(blur_img, cv2.COLOR_BGR2GRAY)\n        orig_gray = cv2.cvtColor(orig_img, cv2.COLOR_BGR2GRAY)\n        orig_gray = clean_target_normalization(orig_gray)\n\n        # Align dimensions if input and clean target differ in resolution (e.g. LongestMaxSize)\n        # Always resize degraded input to match clean target resolution\n        if blur_gray.shape[:2] != orig_gray.shape[:2]:\n            blur_gray = cv2.resize(blur_gray, (orig_gray.shape[1], orig_gray.shape[0]), interpolation=cv2.INTER_CUBIC)\n\n        # Sliding Window Tiling Inference (512x512 with 128 overlap & Gaussian feather blending)\n        restored_cropped = infer_single_image_tiled(\n            model, blur_gray, device, tile_size=args.image_size, overlap=args.overlap\n        )\n\n        # Post-processing: Unsharp Masking\n        restored_final = unsharp_mask(restored_cropped, amount=0.8, radius=1.0)\n\n        # Compute metrics on grayscale images\n        l1 = float(np.mean(np.abs(orig_gray.astype(float) - restored_final.astype(float))))\n        mse = float(np.mean((orig_gray.astype(float) - restored_final.astype(float)) ** 2))\n        psnr = calculate_psnr(orig_gray, restored_final)\n        ssim = calculate_ssim(orig_gray, restored_final)\n\n        # Baseline metrics (degraded vs normalized clean GT)\n        blur_psnr = calculate_psnr(orig_gray, blur_gray)\n        blur_ssim = calculate_ssim(orig_gray, blur_gray)\n\n        check_text = f"{os.path.basename(blur_path)} {blur_level}".lower()\n        is_geometric = any(kw in check_text for kw in GEOMETRIC_DISTORTION_KEYWORDS)\n\n        metrics = {\n            \'split\': split_tag,\n            \'source_id\': source_id,\n            \'blur_level\': blur_level,\n            \'is_geometric\': is_geometric,\n            \'restored_l1\': round(l1, 4),\n            \'restored_mse\': round(mse, 4),\n            \'restored_psnr\': round(psnr, 2),\n            \'restored_ssim\': round(ssim, 4),\n            \'blur_psnr\': round(blur_psnr, 2),\n            \'blur_ssim\': round(blur_ssim, 4),\n            \'psnr_improvement\': round(psnr - blur_psnr, 2),\n        }\n        all_metrics.append(metrics)\n        geom_tag = "GEOM" if is_geometric else "ALIGNED"\n        print(f"  [{split_tag.upper()}:{geom_tag} | {source_id}/{blur_level}] "\n              f"Restored PSNR: {psnr:.2f} dB (Degraded: {blur_psnr:.2f} dB, diff={psnr-blur_psnr:+.2f} dB) | "\n              f"SSIM: {ssim:.4f} (Degraded: {blur_ssim:.4f})")\n\n        # Save images in structured folders\n        sample_name = f"{source_id}_{blur_level}"\n        save_dir = os.path.join(output_dir, sample_name)\n        os.makedirs(save_dir, exist_ok=True)\n        cv2.imwrite(os.path.join(save_dir, \'input_blur.png\'), blur_gray)\n        cv2.imwrite(os.path.join(save_dir, \'restored.png\'), restored_final)\n        cv2.imwrite(os.path.join(save_dir, \'original.png\'), orig_gray)\n\n        # Create comparison contact sheet: Original (GT) | Degraded Input | U-Net Restored\n        target_h = 450\n        def resize_for_display(img, th):\n            hi, wi = img.shape[:2]\n            scale = th / hi\n            return cv2.resize(img, (int(wi * scale), th), interpolation=cv2.INTER_AREA)\n\n        def add_label(img, label):\n            img_c = img.copy()\n            cv2.putText(img_c, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)\n            return img_c\n\n        disp_orig = add_label(resize_for_display(cv2.cvtColor(orig_gray, cv2.COLOR_GRAY2BGR), target_h), "1. Ground Truth")\n        disp_blur = add_label(resize_for_display(cv2.cvtColor(blur_gray, cv2.COLOR_GRAY2BGR), target_h), "2. Degraded Input")\n        disp_rest = add_label(resize_for_display(cv2.cvtColor(restored_final, cv2.COLOR_GRAY2BGR), target_h), "3. U-Net Restored")\n\n        contact_sheet = np.hstack([disp_orig, disp_blur, disp_rest])\n        comparison_path = os.path.join(comp_dir, f"{sample_name}_comparison.png")\n        cv2.imwrite(comparison_path, contact_sheet)\n        print(f"     -> Saved comparison contact sheet: {comparison_path}")\n\n    # ======================== UNPAIRED REAL BLIND EVALUATION (100% PURE U-NET) ========================\n    blind_results = []\n    if blind_samples:\n        print("\\n" + "=" * 68)\n        print("UNPAIRED REAL BLIND EVALUATION (100% PURE U-NET INFERENCE - NO TARGET)")\n        print("=" * 68)\n        for b_item in blind_samples:\n            raw_path = os.path.normpath(b_item[\'input\'])\n            split_tag = b_item.get(\'split\', \'test\')\n            stem = os.path.splitext(os.path.basename(raw_path))[0]\n            raw_bgr = load_image(raw_path)\n            if raw_bgr is None:\n                print(f"  [BLIND-{split_tag.upper()}] SKIP {raw_path}: failed to load")\n                continue\n\n            raw_gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)\n\n            # 100% Pure U-Net Inference directly from raw_gray (NO post-processing, NO unsharp mask, NO fake target)\n            pure_unet_gray = infer_single_image_tiled(\n                model, raw_gray, device, tile_size=args.image_size, overlap=args.overlap, refine_glyphs=False\n            )\n\n            pure_out_path = os.path.join(blind_dir, f"{stem}_pure_unet.png")\n            comp_2col_path = os.path.join(blind_dir, f"{stem}_blind_comparison.png")\n\n            two_col_sheet = create_two_column_blind_comparison(raw_bgr, pure_unet_gray, target_h=680)\n            save_image(pure_out_path, pure_unet_gray)\n            save_image(comp_2col_path, two_col_sheet)\n\n            blind_rec = {\n                \'split\': split_tag,\n                \'source_id\': stem,\n                \'input_path\': raw_path,\n                \'pure_unet_output_path\': pure_out_path,\n                \'comparison_2col_path\': comp_2col_path,\n                \'mode\': \'100% Pure U-Net (Unpaired Blind Evaluation - No Target)\',\n            }\n            blind_results.append(blind_rec)\n            print(\n                f"  [BLIND-{split_tag.upper()} | {stem}] 100% Pure U-Net Output -> {pure_out_path} | "\n                f"2-Column Comparison -> {comp_2col_path}"\n            )\n\n    # Summary separated into pixel_aligned_metrics and geometric_distorted_metrics\n    if all_metrics or blind_results:\n        test_only = [m for m in all_metrics if m.get(\'split\') == \'test\'] or all_metrics\n        pixel_aligned = [m for m in test_only if not m.get(\'is_geometric\', False)]\n        geom_distorted = [m for m in test_only if m.get(\'is_geometric\', False)]\n\n        overall_summary = _summarize_group(test_only)\n        aligned_summary = _summarize_group(pixel_aligned)\n        geom_summary = _summarize_group(geom_distorted)\n\n        print("\\n" + "=" * 68)\n        print("TEST RESULTS SUMMARY (Separated by Alignment)")\n        print("=" * 68)\n        print(f"1. Pixel-Aligned Samples ({aligned_summary[\'count\']} samples - True Pixel Restoration):")\n        print(f"   Avg Restored PSNR: {aligned_summary[\'restored_psnr\']:.2f} dB (Blur: {aligned_summary[\'blur_psnr\']:.2f} dB, Diff: {aligned_summary[\'psnr_improvement\']:+.2f} dB)")\n        print(f"   Avg Restored SSIM: {aligned_summary[\'restored_ssim\']:.4f} (Blur: {aligned_summary[\'blur_ssim\']:.4f}, Diff: {aligned_summary[\'ssim_improvement\']:+.4f})")\n        print("-" * 68)\n        print(f"2. Geometric-Distorted Samples ({geom_summary[\'count\']} samples - Perspective/BookBinding/Optical):")\n        print(f"   Avg Restored PSNR: {geom_summary[\'restored_psnr\']:.2f} dB (Blur: {geom_summary[\'blur_psnr\']:.2f} dB, Diff: {geom_summary[\'psnr_improvement\']:+.2f} dB)")\n        print(f"   Avg Restored SSIM: {geom_summary[\'restored_ssim\']:.4f} (Blur: {geom_summary[\'blur_ssim\']:.4f}, Diff: {geom_summary[\'ssim_improvement\']:+.4f})")\n        print("-" * 68)\n        print(f"3. Overall Paired Test Samples ({overall_summary[\'count\']} test samples, {len(all_metrics)} total incl. val):")\n        print(f"   Avg Restored PSNR: {overall_summary[\'restored_psnr\']:.2f} dB | Avg Restored SSIM: {overall_summary[\'restored_ssim\']:.4f}")\n        print("-" * 68)\n        print(f"4. Unpaired Real Blind Evaluation ({len(blind_results)} samples - 100% Pure U-Net, No Target): {[b[\'source_id\'] for b in blind_results]}")\n        print("=" * 68)\n\n        metrics_path = \'outputs/restoration/test_metrics.json\'\n        os.makedirs(os.path.dirname(metrics_path), exist_ok=True)\n        with open(metrics_path, \'w\', encoding=\'utf-8\') as f:\n            json.dump({\n                \'pixel_aligned_metrics\': aligned_summary,\n                \'geometric_distorted_metrics\': geom_summary,\n                \'average\': overall_summary,\n                \'per_image\': all_metrics,\n                \'blind_real_samples\': blind_results,\n                \'total_samples\': len(all_metrics),\n            }, f, indent=2, ensure_ascii=False)\n        print(f"Metrics saved to: {metrics_path}")\n\n\nif __name__ == \'__main__\':\n    main()\n\n\n'

INFER_RESTORATION_CODE = 'import os\nimport sys\nimport argparse\nimport glob\nimport cv2\nimport numpy as np\nimport torch\n\n# Add parent directory to path to import training modules\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom training.train_restoration import DocumentRestorationUNet\nfrom training.restoration_dataset import (\n    load_image,\n    save_image,\n    unsharp_mask,\n    resize_with_padding,\n    remove_padding,\n    infer_single_image_tiled,\n)\n\n\ndef restore_image_for_yolov8(\n    model: torch.nn.Module,\n    img_bgr: np.ndarray,\n    device: torch.device,\n    tile_size: int = 512,\n    overlap: int = 128,\n    preserve_color: bool = False,\n    refine_glyphs: bool = False,\n    apply_postprocess: bool = False,\n) -> np.ndarray:\n    """\n    AI-Centric Document Restoration for YOLOv8:\n    1. Passes image directly through `DocumentRestorationUNet` (`infer_single_image_tiled`).\n    2. Default mode (preserve_color=False, apply_postprocess=False):\n       Output is 100% Pure U-Net output tensor converted to BGR uint8,\n       WITHOUT any background clipping (e.g. np.where(y < 245, ..., 255)) or unsharp masking.\n    3. If preserve_color=True: merges U-Net\'s raw prediction with original chrominance (Cr, Cb).\n    4. If apply_postprocess=True: optionally applies stroke unsharp masking and white floor clipping.\n    """\n    if img_bgr.ndim == 2:\n        gray = img_bgr\n        img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)\n    else:\n        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)\n\n    # 100% Pure U-Net forward pass\n    y_restored = infer_single_image_tiled(\n        model, gray, device, tile_size=tile_size, overlap=overlap, refine_glyphs=refine_glyphs\n    )\n\n    if apply_postprocess:\n        y_proc = np.where(y_restored < 245, unsharp_mask(y_restored, amount=0.65, radius=1.0), 255).astype(np.uint8)\n    else:\n        y_proc = y_restored\n\n    if not preserve_color:\n        return cv2.cvtColor(y_proc, cv2.COLOR_GRAY2BGR)\n\n    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)\n    _, cr_chan, cb_chan = cv2.split(ycrcb)\n    chroma_sat = np.hypot(cr_chan.astype(np.float32) - 128.0, cb_chan.astype(np.float32) - 128.0)\n    color_fg_mask = (y_proc < 245) & (chroma_sat > 18.0)\n    cr_clean = np.where(color_fg_mask, cr_chan, 128).astype(np.uint8)\n    cb_clean = np.where(color_fg_mask, cb_chan, 128).astype(np.uint8)\n    return cv2.cvtColor(cv2.merge([y_proc, cr_clean, cb_clean]), cv2.COLOR_YCrCb2BGR)\n\n\ndef create_three_column_preview(\n    raw_bgr: np.ndarray,\n    target_gray_or_bgr: np.ndarray,\n    unet_bgr: np.ndarray,\n    display_height: int = 680,\n) -> np.ndarray:\n    """\n    Build a 3-column comparison sheet:\n    [1. Anh Real Goc (Input) | 2. Dap an Mau (Target) | 3. Ket qua tu Model U-Net (Output cho YOLO)]\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    if target_gray_or_bgr.ndim == 2:\n        target_bgr = cv2.cvtColor(target_gray_or_bgr, cv2.COLOR_GRAY2BGR)\n    else:\n        target_bgr = target_gray_or_bgr.copy()\n    if unet_bgr.ndim == 2:\n        unet_bgr = cv2.cvtColor(unet_bgr, cv2.COLOR_GRAY2BGR)\n\n    def _resize_col(img: np.ndarray, th: int) -> np.ndarray:\n        h, w = img.shape[:2]\n        scale = th / float(max(h, 1))\n        nw = max(220, int(w * scale))\n        return cv2.resize(img, (nw, th), interpolation=cv2.INTER_AREA)\n\n    def _add_header(img: np.ndarray, title: str, color: tuple) -> np.ndarray:\n        header_h = 44\n        canvas = np.full((img.shape[0] + header_h, img.shape[1], 3), 245, dtype=np.uint8)\n        canvas[header_h:, :] = img\n        cv2.putText(canvas, title, (10, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.58, color, 2, cv2.LINE_AA)\n        return canvas\n\n    col1 = _add_header(_resize_col(raw_bgr, display_height), "1. Anh Real Goc (Input)", (0, 0, 210))\n    col2 = _add_header(_resize_col(target_bgr, display_height), "2. Dap an Mau (Target)", (0, 140, 0))\n    col3 = _add_header(_resize_col(unet_bgr, display_height), "3. Ket qua tu Model U-Net (Output cho YOLO)", (180, 60, 0))\n\n    # Match column widths if needed\n    divider = np.full((col1.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col1, divider, col2, divider, col3])\n\n\ndef create_two_column_blind_preview(\n    raw_bgr: np.ndarray,\n    unet_bgr: np.ndarray,\n    display_height: int = 680,\n) -> np.ndarray:\n    """\n    Build an honest 2-column blind evaluation sheet for unpaired real images (e.g., real_11.jpg, real_12.jpg):\n    [1. Anh Real Goc (Input duy nhat) | 2. Ket qua 100% Pure U-Net (Output tu AI)]\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    if unet_bgr.ndim == 2:\n        unet_bgr = cv2.cvtColor(unet_bgr, cv2.COLOR_GRAY2BGR)\n\n    def _resize_col(img: np.ndarray, th: int) -> np.ndarray:\n        h, w = img.shape[:2]\n        scale = th / float(max(h, 1))\n        nw = max(240, int(w * scale))\n        return cv2.resize(img, (nw, th), interpolation=cv2.INTER_AREA)\n\n    def _add_header(img: np.ndarray, title: str, color: tuple) -> np.ndarray:\n        header_h = 44\n        canvas = np.full((img.shape[0] + header_h, img.shape[1], 3), 245, dtype=np.uint8)\n        canvas[header_h:, :] = img\n        cv2.putText(canvas, title, (10, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.58, color, 2, cv2.LINE_AA)\n        return canvas\n\n    col1 = _add_header(\n        _resize_col(raw_bgr, display_height),\n        "1. Anh Real Goc (Input duy nhat)",\n        (0, 0, 210),\n    )\n    col2 = _add_header(\n        _resize_col(unet_bgr, display_height),\n        "2. Ket qua 100% Pure U-Net (Output tu AI)",\n        (180, 60, 0),\n    )\n\n    divider = np.full((col1.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col1, divider, col2])\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Run DocumentRestorationUNet Inference for YOLOv8")\n    parser.add_argument("--model", type=str, default="models/restoration_best.pth")\n    parser.add_argument("--input-dir", "--input", dest="input_dir", type=str, default="datasets/real")\n    parser.add_argument("--target-dir", type=str, default="datasets/processed/real_pairs/target")\n    parser.add_argument("--output-dir", "--output", dest="output_dir", type=str, default="outputs/restoration_real")\n    parser.add_argument("--preview-dir", type=str, default="datasets/real/unet_results_preview")\n    parser.add_argument("--blind-out-dir", type=str, default="outputs/restoration/real_blind_test")\n    parser.add_argument("--tile-size", type=int, default=512, help="Sliding window tile size")\n    parser.add_argument("--overlap", type=int, default=128, help="Overlap between tiles")\n    parser.add_argument(\n        "--preserve-color",\n        action="store_true",\n        default=False,\n        help="Preserve foreground color via YCrCb space while restoring luminance Y with U-Net (default: False for Pure U-Net Grayscale)",\n    )\n    parser.add_argument(\n        "--postprocess",\n        action="store_true",\n        default=False,\n        help="Apply stroke unsharp mask and background thresholding (default: False for Pure U-Net)",\n    )\n    args = parser.parse_args()\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Device: {device}")\n\n    model = DocumentRestorationUNet().to(device)\n    if os.path.exists(args.model):\n        model.load_state_dict(torch.load(args.model, map_location=device, weights_only=True))\n        print(f"Loaded model from {args.model}")\n    else:\n        print(f"ERROR: Model not found at {args.model}.")\n        return\n\n    model.eval()\n\n    os.makedirs(args.output_dir, exist_ok=True)\n    os.makedirs(args.preview_dir, exist_ok=True)\n    os.makedirs(args.blind_out_dir, exist_ok=True)\n    comp_dir = os.path.join(args.output_dir, "comparison")\n    os.makedirs(comp_dir, exist_ok=True)\n\n    extensions = (".jpg", ".jpeg", ".png", ".webp", ".jfif")\n    image_paths = []\n    if os.path.exists(args.input_dir):\n        for entry in sorted(os.listdir(args.input_dir)):\n            full_p = os.path.join(args.input_dir, entry)\n            if os.path.isfile(full_p) and os.path.splitext(entry)[1].lower() in extensions:\n                image_paths.append(full_p)\n\n    print(f"Found {len(image_paths)} images in {args.input_dir}")\n\n    processed = 0\n    for idx, img_path in enumerate(image_paths, 1):\n        raw_bgr = load_image(img_path)\n        if raw_bgr is None:\n            print(f"  [{idx}/{len(image_paths)}] SKIP: Could not load image {img_path}")\n            continue\n\n        base_name = os.path.splitext(os.path.basename(img_path))[0]\n        is_blind_eval = base_name.lower() in ("real_11", "real_12")\n\n        # For blind evaluation images (real_11, real_12), strictly enforce 100% Pure U-Net\n        restored_bgr = restore_image_for_yolov8(\n            model,\n            raw_bgr,\n            device,\n            tile_size=args.tile_size,\n            overlap=args.overlap,\n            preserve_color=False if is_blind_eval else args.preserve_color,\n            refine_glyphs=False,\n            apply_postprocess=False if is_blind_eval else args.postprocess,\n        )\n\n        filename = f"{base_name}_unet_yolo.png"\n        save_image(os.path.join(args.output_dir, filename), restored_bgr)\n\n        target_path = os.path.join(args.target_dir, f"{base_name}.png")\n        has_valid_target = (not is_blind_eval) and os.path.exists(target_path)\n\n        if has_valid_target:\n            target_img = load_image(target_path)\n            preview_sheet = create_three_column_preview(raw_bgr, target_img, restored_bgr)\n        else:\n            preview_sheet = create_two_column_blind_preview(raw_bgr, restored_bgr)\n            save_image(os.path.join(args.blind_out_dir, f"{base_name}_pure_unet.png"), restored_bgr)\n            save_image(os.path.join(args.blind_out_dir, f"{base_name}_blind_comparison.png"), preview_sheet)\n\n        preview_path = os.path.join(args.preview_dir, f"{base_name}_unet_preview.png")\n        save_image(preview_path, preview_sheet)\n        save_image(os.path.join(comp_dir, f"comparison_{idx}_{base_name}.png"), preview_sheet)\n\n        processed += 1\n        mode_tag = "2-Col Blind Pure U-Net" if not has_valid_target else "3-Col Paired Preview"\n        print(f"  [{idx}/{len(image_paths)}] [{mode_tag}] Processed -> {preview_path}")\n\n    print(f"\\nFinished processing {processed} images.")\n    print(f"Restored YOLOv8 images: {args.output_dir}")\n    print(f"U-Net Previews: {args.preview_dir}")\n    print(f"Blind Test Pure U-Net Previews: {args.blind_out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n\n'

for fname, code in [
    ("restoration_dataset.py", RESTORATION_DATASET_CODE),
    ("train_restoration.py", TRAIN_RESTORATION_CODE),
    ("evaluate_restoration.py", EVALUATE_RESTORATION_CODE),
    ("infer_restoration.py", INFER_RESTORATION_CODE),
]:
    dst = f"{PROJECT_DIR}/training/{fname}"
    with open(dst, "w", encoding="utf-8") as f:
        f.write(code)
    print(f"  [OK] Updated {dst} ({len(code):,} chars)")

sys.path.insert(0, PROJECT_DIR)


In [ ]:
#@title 4. Kiểm tra dữ liệu đã gộp: train.csv (267 mẫu = 183 gốc + 28 real_pairs x3), val.csv (24 mẫu có target + real_12.jpg blind), test.csv (16 mẫu có target + real_11.jpg blind)
if not os.path.exists(f'{DATASET_ROOT}/processed/real_pairs/target/real_20.png'):
    print('Đang tạo cặp dữ liệu thực tế real_pairs (loại trừ real_11 & real_12) ...')
    prepare_real_pairs()

df_manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
df_train    = pd.read_csv(TRAIN_CSV, dtype=str, keep_default_na=False)
df_val      = pd.read_csv(VAL_CSV, dtype=str, keep_default_na=False)
df_test     = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)

print(f'Tổng số dòng trong dataset_manifest.csv: {len(df_manifest)}')
print(f'  - train.csv : {len(df_train)} mẫu (183 mẫu gốc + 84 mẫu từ 28 cặp real_pairs oversample x3)')
print(f'  - val.csv   : {len(df_val)} dòng (24 mẫu có target + 1 ảnh blind val real_12.jpg)')
print(f'  - test.csv  : {len(df_test)} dòng (16 mẫu có target + 1 ảnh blind test real_11.jpg)')

# Kiểm tra chống rò rỉ dữ liệu (Zero Data Leakage cho real_11 và real_12)
train_inputs_joined = ' '.join(df_train['input'].tolist()).lower()
assert 'real_11' not in train_inputs_joined and 'real_12' not in train_inputs_joined, 'LỖI DATA LEAKAGE: real_11 hoặc real_12 xuất hiện trong train.csv!'
assert len(df_train) >= 261, f'Kỳ vọng >= 261 mẫu train (183 gốc + 26..28 real_pairs x3), thực tế: {len(df_train)}'
assert len(df_val) >= 24, f'Kỳ vọng >= 24 dòng val, thực tế: {len(df_val)}'
assert len(df_test) in (16, 17), f'Kỳ vọng 16 hoặc 17 dòng test (gồm real_11.jpg), thực tế: {len(df_test)}'
assert os.path.exists(f'{DATASET_ROOT}/processed/test/input/real_11.jpg'), 'Thiếu file datasets/processed/test/input/real_11.jpg'
assert os.path.exists(f'{DATASET_ROOT}/processed/val/input/real_12.jpg'), 'Thiếu file datasets/processed/val/input/real_12.jpg'

print('\n[OK] Kiểm tra chống Data Leakage: real_11.jpg và real_12.jpg hoàn toàn độc lập khỏi tập Train!')
print('\nThống kê theo split trong manifest:')
print(df_manifest.groupby('split').agg(
    inputs=('input_path', 'count'),
    unique_targets=('target_path', 'nunique'),
))

HAS_VAL  = len(df_val) > 0
HAS_TEST = len(df_test) > 0
MODE = 'full' if (HAS_VAL and HAS_TEST) else 'smoke'


In [ ]:
#@title 5. Khởi tạo Datasets & DataLoaders (Tự động lọc biến dạng hình học & tách riêng ảnh Unpaired Blind Evaluation)
train_ds = DocumentRestorationDataset(
    TRAIN_CSV,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    patch_size=(IMAGE_SIZE, IMAGE_SIZE),
    split='train',
    is_train=True,
)
val_ds = DocumentRestorationDataset(
    VAL_CSV,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    patch_size=(IMAGE_SIZE, IMAGE_SIZE),
    split='val',
    is_train=False,
) if HAS_VAL else None
test_ds = DocumentRestorationDataset(
    TEST_CSV,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    patch_size=(IMAGE_SIZE, IMAGE_SIZE),
    split='test',
    is_train=False,
) if HAS_TEST else None

assert train_ds.missing_count == 0, f'Lỗi missing file trong train.csv: {train_ds.missing_count}'
assert val_ds is None or val_ds.missing_count == 0, f'Lỗi missing file trong val.csv: {val_ds.missing_count}'
assert test_ds is None or test_ds.missing_count == 0, f'Lỗi missing file trong test.csv: {test_ds.missing_count}'
assert len(train_ds) > 0, 'Không tìm thấy cặp ảnh train hợp lệ!'

print(f'[TRAIN] Tổng CSV: {train_ds.total_rows} | Đã lọc hình học/Anh06: {train_ds.filtered_count} | Missing: {train_ds.missing_count} -> Sử dụng: {len(train_ds)} cặp chuẩn')
if val_ds:
    print(f'[VAL]   Tổng CSV: {val_ds.total_rows} | Đã lọc hình học: {val_ds.filtered_count} | Unpaired Blind (real_12): {val_ds.unpaired_count} | Missing: {val_ds.missing_count} -> Sử dụng tính PSNR: {len(val_ds)} cặp chuẩn')
if test_ds:
    print(f'[TEST]  Tổng CSV: {test_ds.total_rows} | Unpaired Blind (real_11): {test_ds.unpaired_count} | Missing: {test_ds.missing_count} -> Sử dụng tính PSNR: {len(test_ds)} cặp chuẩn')

sample_x, sample_y = train_ds[0]
print(f'\nTensor shape: input={tuple(sample_x.shape)}, target={tuple(sample_y.shape)} '
      f'(Target mean={sample_y.mean():.3f}, max={sample_y.max():.2f} — nền giấy đã chuẩn hóa trắng tinh)')

kw = dict(num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw) if val_ds else None
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw) if test_ds else None

VIS_DS = val_ds if (val_ds is not None and len(val_ds) > 0) else train_ds
VIS_IDX = list(range(min(3, len(VIS_DS))))


In [ ]:
#@title 6. Khởi tạo Model U-Net v5 & DocumentCleanLoss v5 + Optimizer CosineAnnealingLR
from training.train_restoration import DocumentRestorationUNet, DocumentCleanLoss
import torch.optim as optim

model = DocumentRestorationUNet(in_channels=1, out_channels=1).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model DocumentRestorationUNet v5 instantiated on {DEVICE} (Parameters: {total_params:,})")

loss_fn = DocumentCleanLoss().to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

# Nạp warm-start nếu có
if os.path.exists(BEST):
    try:
        sd = torch.load(BEST, map_location=DEVICE, weights_only=True)
        model.load_state_dict(sd, strict=False)
        print(f"Loaded weights from {BEST} into U-Net v5.")
    except Exception as e:
        print(f"Checkpoint {BEST} loading note: {e}. Starting fresh v5 training.")


In [ ]:
#@title 7. Hàm đánh giá & Hiển thị trực quan ảnh Grayscale chuẩn [0, 1]
@torch.no_grad()
def evaluate(loader):
    model.eval(); tl = tp = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        p = model(x)
        tl += loss_fn(p, y).item(); tp += calculate_psnr(p, y); n += 1
    return tl / max(n, 1), tp / max(n, 1)

def to_gray_img(t):
    return t.detach().cpu().squeeze().clamp(0, 1).numpy()

@torch.no_grad()
def show_samples(title):
    model.eval()
    fig, ax = plt.subplots(len(VIS_IDX), 3, figsize=(12, 4 * len(VIS_IDX)), squeeze=False)
    for r, i in enumerate(VIS_IDX):
        x, y = VIS_DS[i]
        p = model(x.unsqueeze(0).to(DEVICE))[0]
        in_psnr = calculate_psnr(x, y)
        out_psnr = calculate_psnr(p.cpu(), y)
        cols = [
            (y, '1. Ground Truth (Clean White)'),
            (x, f'2. Degraded Input ({in_psnr:.2f} dB)'),
            (p, f'3. U-Net Restored ({out_psnr:.2f} dB)'),
        ]
        for c, (im, name) in enumerate(cols):
            ax[r][c].imshow(to_gray_img(im), cmap='gray', vmin=0.0, vmax=1.0)
            ax[r][c].set_title(name)
            ax[r][c].axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_history():
    if not history:
        return
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(h.epoch, h.train_loss, label='Train Loss', marker='o', markersize=3)
    if 'val_loss' in h and h.val_loss.notna().any():
        ax[0].plot(h.epoch, h.val_loss, label='Val Loss', marker='s', markersize=3)
    ax[0].set_title('DocumentCleanLoss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(h.epoch, h.train_psnr, label='Train PSNR', marker='o', markersize=3)
    if 'val_psnr' in h and h.val_psnr.notna().any():
        ax[1].plot(h.epoch, h.val_psnr, label='Val PSNR', marker='s', markersize=3)
    ax[1].set_title('PSNR (dB)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 8. Huấn luyện Model trên GPU (Tự động lưu Best Checkpoint vào Drive)
from IPython.display import clear_output

# Số vòng lặp random crop mỗi epoch
REPEAT_PER_EPOCH = 2 #@param {type:"integer"}

for epoch in range(start_epoch, EPOCHS + 1):
    model.train(); tl = tp = n = 0; t0 = time.time()
    for _ in range(REPEAT_PER_EPOCH):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                p = model(x)
            loss = loss_fn(p.float(), y.float())
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tl += loss.item()
            tp += calculate_psnr(p.detach().float(), y.float())
            n += 1
    scheduler.step()

    rec = dict(
        epoch=epoch,
        train_loss=round(tl / max(n, 1), 6),
        train_psnr=round(tp / max(n, 1), 2),
        val_loss=None,
        val_psnr=None,
        lr=round(opt.param_groups[0]['lr'], 7),
        seconds=round(time.time() - t0, 1),
    )
    if val_loader:
        vl, vp = evaluate(val_loader)
        rec['val_loss'], rec['val_psnr'] = round(vl, 6), round(vp, 2)
    history.append(rec)

    torch.save(model.state_dict(), LAST)
    shutil.copyfile(LAST, f'{MODELS_DIR}/restoration_last.pth')

    score = rec['val_psnr'] if val_loader else rec['train_psnr']
    if score > best_psnr:
        best_psnr = score
        torch.save(model.state_dict(), BEST)
        shutil.copyfile(BEST, f'{MODELS_DIR}/restoration_best.pth')

    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), history=history,
                    epoch=epoch, best_psnr=best_psnr), f'{RUN_DIR}/resume.pt')
    pd.DataFrame(history).to_csv(LOG_CSV, index=False)
    log_payload = dict(
        device=str(DEVICE),
        mode=MODE,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        epochs=epoch,
        train_samples=len(train_ds),
        val_samples=len(val_ds) if val_ds else 0,
        best_val_psnr=round(best_psnr, 2),
        log=history,
    )
    with open(LOG_JSON, 'w', encoding='utf-8') as f:
        json.dump(log_payload, f, indent=2)
    shutil.copyfile(LOG_JSON, f'{MODELS_DIR}/training_log.json')

    print(f"Epoch [{epoch}/{EPOCHS}] ({rec['seconds']}s) | Train Loss: {rec['train_loss']:.4f} | Train PSNR: {rec['train_psnr']:.2f} dB"
          + (f" | Val Loss: {rec['val_loss']:.4f} | Val PSNR: {rec['val_psnr']:.2f} dB (Best: {best_psnr:.2f} dB)" if val_loader else ''))
    if epoch % VIS_EVERY == 0 or epoch == EPOCHS:
        clear_output(wait=True)
        plot_history()
        show_samples(f'Epoch {epoch}/{EPOCHS} — Best PSNR: {best_psnr:.2f} dB')

print('Huấn luyện hoàn tất! Checkpoint tốt nhất đã lưu tại:', BEST)


In [ ]:
#@title 9. Đánh giá Tập Test & Val bằng evaluate_restoration.py
BLIND_OUT_DIR = f'{PROJECT_DIR}/outputs/restoration/real_blind_test'
get_ipython().system(f'python training/evaluate_restoration.py --model "{BEST}" --test-csv "{TEST_CSV}" --val-csv "{VAL_CSV}" --include-val --blind-out-dir "{BLIND_OUT_DIR}"')


In [ ]:
#@title 10. Suy luận Trực quan Toàn bộ Ảnh Thực Tế trong Val/Test & Tự động Lưu Weights về Google Drive
import matplotlib.pyplot as plt
import cv2
from training.restoration_dataset import load_image, infer_single_image_tiled

print("=== CHẠY SUY LUẬN U-NET V5 TRÊN CÁC ẢNH THỰC TẾ (VAL & TEST) ===")

eval_images = [
    # Tập Val
    ("VAL: real_12 (Bóng râm / Chiếu sáng chênh lệch)", f"{DATASET_ROOT}/processed/val/input/real_12.jpg"),
    ("VAL: real_04 (Giấy ngả vàng / Nếp gập trang)", f"{DATASET_ROOT}/processed/val/input/real_04.jpg"),
    ("VAL: real_25 (Thấu mực mặt sau / Giấy mỏng)", f"{DATASET_ROOT}/processed/val/input/real_25.jpg"),
    ("VAL: IMG20261022100617 (Camera điện thoại thực tế)", f"{DATASET_ROOT}/processed/val/input/IMG20261022100617.jpg"),
    # Tập Test
    ("TEST: real_11 (Bóng lóa flash / Gradient sáng gắt)", f"{DATASET_ROOT}/processed/test/input/real_11.jpg"),
    ("TEST: real_02 (Nhiễu ISO hạt tiêu / Rung tay)", f"{DATASET_ROOT}/processed/test/input/real_02.jpg"),
    ("TEST: real_29 (Thấu mực đậm / Con dấu màu)", f"{DATASET_ROOT}/processed/test/input/real_29.jpg"),
    ("TEST: IMG20261022101140 (Chụp phòng tối / Nhiễu hạt)", f"{DATASET_ROOT}/processed/test/input/IMG20261022101140.jpg"),
]

existing_eval = [(title, p) for title, p in eval_images if os.path.exists(p)]
print(f"Tìm thấy {len(existing_eval)} ảnh thực tế trong Val & Test để đánh giá.")

if existing_eval:
    model.eval()
    plt.figure(figsize=(16, 4.5 * len(existing_eval)))
    plot_idx = 1
    
    for title, img_path in existing_eval:
        img_bgr = load_image(img_path)
        if img_bgr is None:
            continue
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr
        
        # Pure U-Net v5 Inference
        with torch.no_grad():
            restored = infer_single_image_tiled(model, gray, DEVICE, tile_size=512, overlap=128)
        
        # Ảnh Gốc
        plt.subplot(len(existing_eval), 2, plot_idx)
        plt.imshow(gray, cmap="gray", vmin=0, vmax=255)
        plt.title(f"[ẢNH THỰC TẾ GỐC]\n{title}", fontsize=11, fontweight="bold")
        plt.axis("off")
        
        # Ảnh U-Net v5 Khôi Phục
        plt.subplot(len(existing_eval), 2, plot_idx + 1)
        plt.imshow(restored, cmap="gray", vmin=0, vmax=255)
        plt.title(f"[U-NET v5 KHÔI PHỤC]\n{title}", fontsize=11, fontweight="bold", color="darkgreen")
        plt.axis("off")
        
        plot_idx += 2
        
    plt.tight_layout()
    plt.show()

# Tự động sao lưu restoration_best.pth về thư mục gốc Google Drive
DRIVE_BACKUP_DIR = f"{PROJECT_DIR}/models"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
if os.path.exists(BEST):
    drive_dest = f"{DRIVE_BACKUP_DIR}/restoration_best.pth"
    shutil.copy2(BEST, drive_dest)
    print(f"\n[THÀNH CÔNG] Đã tự động sao lưu trọng số tốt nhất về Google Drive:")
    print(f"  -> {drive_dest} ({os.path.getsize(drive_dest)/1024/1024:.1f} MB)")


In [ ]:
#@title 11. Tải trọng số tốt nhất (restoration_best.pth) trực tiếp về máy tính
from google.colab import files
for f in [BEST, LOG_JSON]:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)
